<a href="https://colab.research.google.com/github/chembiolab/chembiolab.github.io/blob/main/HTRA1_diversify_revalidate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HTRA1 서열 다양화 + 재검증 — 2026-07-27

기존 backbone(L06/L09/L12)에서 **MPNN 8서열/backbone** 뽑아 → target 고정 폴딩 → 판정.
파일럿 = 각 길이 상위 N backbone (기본 3 → 9 backbone × 8 = 72 폴딩).

**실행 계획: 자기 전에 셀 0~3을 돌려 manifest가 맞는지 확인(빠름), 그다음 셀 4(긴 폴딩)를 시작하고 잔다.**
셀 4는 서열마다 Drive에 즉시 저장 + 이미 있으면 skip이라, 런타임 끊겨도 아침에 셀 4만 다시 돌리면 남은 것만 이어 접는다(CU 재낭비 없음).

**오늘(07-27) 검증된 함정 대비 전부 포함:**
- 번호 충돌 제거(binder 1.. / target 1000.., 2000..) — 안 하면 ColabDesign이 chain 뒤집어 binder 32Å 날아감
- MPNN 슬라이스는 `seq.split("/")[0]`(binder=첫 chain) — `[:nres]` 함정 회피. 고유서열 수 검증
- binder 이동 자가검증(<8Å), Drive 즉시저장, motif 판정(protomer 합쳐져 나와도 처리)

In [ ]:
#@title 0. 설치 + Drive
import os
try: import colabdesign
except Exception: os.system('pip -q install git+https://github.com/sokrypton/ColabDesign.git')
os.system('pip -q install gemmi 2>/dev/null')
if not os.path.isfile('/content/params/params_model_1_ptm.npz'):
    os.makedirs('/content/params',exist_ok=True); os.system('apt-get -qq install aria2 >/dev/null 2>&1')
    os.system('aria2c -q -x8 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar -d /content/params')
    os.system('tar -xf /content/params/alphafold_params_2022-12-06.tar -C /content/params')
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
import gemmi
print('params:',os.path.isfile('/content/params/params_model_1_ptm.npz'),'| drive:',os.path.isdir('/content/drive/MyDrive'),'| gemmi:',gemmi.__version__)

Mounted at /content/drive
params: True | drive: True | gemmi: 0.7.5


In [ ]:
import glob
outs=glob.glob('/content/drive/MyDrive/htra1_v3_short/diversify_out/*.pdb')
print("폴딩된 pdb:", len(outs), "개")

폴딩된 pdb: 585 개


In [ ]:
import os, subprocess
CJ_DRIVE='/content/drive/MyDrive/htra1_v3_short/clamp_judge.py'
print("Drive에 clamp_judge:", os.path.isfile(CJ_DRIVE))
print("content에 clamp_judge:", os.path.isfile('/content/clamp_judge.py'))
if os.path.isfile(CJ_DRIVE):
    subprocess.run(['cp',CJ_DRIVE,'/content/clamp_judge.py'])
    print("복사됨")

Drive에 clamp_judge: True
content에 clamp_judge: False
복사됨


In [ ]:
#@title 1. judge 기록
_J = r'''
import os, sys, glob, argparse, csv
import numpy as np
RMSD_CUT, CONTACT_CUT, CATALYTIC_CUT = 2.0, 4.5, 8.0
JUNCTION_LO, JUNCTION_HI = 8.0, 16.0
BINDER_MAX = 40
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
HIS220,SER328,L3_GLU,LD_A,LD_B=220,328,306,287,290

def parse_pdb(path):
    ch={};seen=set()
    for ln in open(path):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        c=ln[21];rn=ln[17:20].strip()
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        k=(c,rs)
        if k not in seen: seen.add(k); ch.setdefault(c,{})[rs]=[rs,rn,{}]
        ch[c][rs][2][an]=np.array([x,y,z])
    return {c:[d[k] for k in sorted(d)] for c,d in ch.items()}

def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def ca(res): return res[2].get("CA")
def hc(res): return list(res[2].values())
def md(ba,res):
    if res is None: return 1e9
    t=hc(res)
    if not t: return 1e9
    B=np.array(ba);T=np.array(t)
    return float(np.sqrt(((B[:,None,:]-T[None,:,:])**2).sum(-1)).min())
def kab(P,Q):
    Pc=P-P.mean(0);Qc=Q-Q.mean(0);H=Pc.T@Qc;U,S,Vt=np.linalg.svd(H)
    d=np.sign(np.linalg.det(Vt.T@U.T));R=Vt.T@np.diag([1,1,d])@U.T
    return float(np.sqrt((((R@Pc.T).T-Qc)**2).sum()/len(P)))
def prmsd(pc,poff,rc,roff):
    pm={r[0]-poff:ca(r) for r in pc if ca(r) is not None}
    rm={r[0]-roff:ca(r) for r in rc if ca(r) is not None}
    common=sorted(set(pm)&set(rm))
    if len(common)<4: return 1e9
    return kab(np.array([pm[a] for a in common]),np.array([rm[a] for a in common]))

def find_all(s, motif):
    out=[];i=s.find(motif)
    while i!=-1: out.append(i); i=s.find(motif,i+1)
    return out

def segment_protomers(target_chains):
    """target_chains: list of residue-lists (one per target chain). Each protease protomer
    has one NAHV(His220) and one GNSGG(Ser328). Separate chains -> each chain is a protomer.
    Merged chain (>=2 NAHV) -> split between Ser328_k and His220_(k+1). Robust to either."""
    prot=[]
    for rl in target_chains:
        seq=o1(rl)
        nahv=find_all(seq,"NAHV"); gnsgg=find_all(seq,"GNSGG")
        if not nahv: continue
        if len(nahv)==1:
            prot.append(rl)
        else:
            bnds=[0]
            for k in range(len(nahv)-1):
                g=gnsgg[k] if k<len(gnsgg) else nahv[k]      # Ser328 of protomer k
                bnds.append((g+nahv[k+1])//2)                # boundary after Ser328_k, before His220_(k+1)
            bnds.append(len(rl))
            for k in range(len(nahv)):
                prot.append(rl[bnds[k]:bnds[k+1]])
    return prot if len(prot)>=2 else None

def resolve(rl, k):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: raise RuntimeError(f"[P{k}] motif missing after segmentation")
    hr=rl[ih+2][0];sr=rl[iss+2][0];off=hr-HIS220
    if off!=sr-SER328: raise RuntimeError(f"[P{k}] gap His/Ser")
    def anc(a,exp,lab):
        rs=a+off;hit=[r for r in rl if r[0]==rs]
        if not hit: raise RuntimeError(f"[P{k}] {lab} absent")
        rn=hit[0][1];ok=rn in exp
        print(f"  [ANCHOR P{k}] auth{a}->{rs}={rn} ({'OK' if ok else 'MISMATCH'}) [{lab}]")
        if not ok: raise RuntimeError(f"[P{k}] {lab}={rn}")
        return hit[0]
    print(f"  [OFFSET P{k}] offset={off}")
    anc(HIS220,{"HIS"},"His220");anc(SER328,{"SER"},"Ser328");anc(L3_GLU,{"GLU"},"L3-306")
    anc(LD_A,{"SER","THR"},"LD-287");anc(LD_B,{"ASN"},"LD-290")
    return off
def rba(rl,a,off):
    for r in rl:
        if r[0]==a+off: return r
    return None

def judge_one(pp, ref_prot, ref_offs, tag):
    print(f"\n=== {tag} ===")
    ch=parse_pdb(pp)
    if not ch: return {"tag":tag,"error":"empty_pdb"}
    binder=min(ch, key=lambda c:len(ch[c]))
    if len(ch[binder])>BINDER_MAX:
        return {"tag":tag,"error":f"no short binder chain (shortest={len(ch[binder])})"}
    target_chains=[ch[c] for c in ch if c!=binder]
    ba=[a for r in ch[binder] for a in hc(r)]
    prot=segment_protomers(target_chains)
    if prot is None: return {"tag":tag,"error":"found <2 His220(NAHV) protomers"}
    offs=[resolve(p,k) for k,p in enumerate(prot)]
    tr=max(min(prmsd(p,po,rp,ro) for rp,ro in zip(ref_prot,ref_offs)) for p,po in zip(prot,offs))
    per=[]
    for p,off in zip(prot,offs):
        per.append({"d306":md(ba,rba(p,L3_GLU,off)),"dLD":min(md(ba,rba(p,LD_A,off)),md(ba,rba(p,LD_B,off))),
            "dHis":md(ba,rba(p,HIS220,off)),"dSer":md(ba,rba(p,SER328,off)),
            "r306":rba(p,L3_GLU,off),"r287":rba(p,LD_A,off),"r290":rba(p,LD_B,off)})
    jp=False;jg=1e9
    for i,X in enumerate(per):
        if X["d306"]>CONTACT_CUT: continue
        for jx,Y in enumerate(per):
            if i==jx or Y["dLD"]>CONTACT_CUT: continue
            ld=Y["r287"] if md(ba,Y["r287"])<=md(ba,Y["r290"]) else Y["r290"]
            if X["r306"] is None or ld is None: continue
            cx=ca(X["r306"]);cy=ca(ld)
            if cx is None or cy is None: continue
            g=float(np.linalg.norm(cx-cy))
            if JUNCTION_LO<=g<=JUNCTION_HI: jp=True;jg=g
    cc=all(p["dHis"]>CATALYTIC_CUT and p["dSer"]>CATALYTIC_CUT for p in per)
    mc=min(min(p["dHis"],p["dSer"]) for p in per)
    return {"tag":tag,"target_rmsd":round(tr,2),"collapse_ok":tr<=RMSD_CUT,
        "min_d306":round(min(p["d306"] for p in per),2),"min_dLD":round(min(p["dLD"] for p in per),2),
        "junction_pass":jp,"junction_geom":round(jg,2) if jg<1e8 else "","min_catalytic":round(mc,2),
        "catalytic_clear":cc,"clean_clamp":tr<=RMSD_CUT and jp and cc,"error":""}

def main():
    ap=argparse.ArgumentParser();ap.add_argument("--pred_dir",required=True);ap.add_argument("--ref_pdb",required=True);ap.add_argument("--out_csv",default="revalid.csv")
    a=ap.parse_args()
    refch=parse_pdb(a.ref_pdb)
    ref_chains=[refch[c] for c in refch if len(refch[c])>BINDER_MAX]
    print("[REF] segment protomers")
    ref_prot=segment_protomers(ref_chains)
    if ref_prot is None: sys.exit("reference: <2 protomers found")
    ref_offs=[resolve(p,k) for k,p in enumerate(ref_prot)]
    res=[]
    for p in sorted(glob.glob(os.path.join(a.pred_dir,"*.pdb"))):
        tag=os.path.splitext(os.path.basename(p))[0]
        try: r=judge_one(p,ref_prot,ref_offs,tag)
        except RuntimeError as e: r={"tag":tag,"error":str(e)}
        res.append(r)
    f=["tag","target_rmsd","collapse_ok","min_d306","min_dLD","junction_pass","junction_geom","min_catalytic","catalytic_clear","clean_clamp","error"]
    with open(a.out_csv,"w",newline="") as fh:
        w=csv.DictWriter(fh,fieldnames=f,extrasaction="ignore");w.writeheader()
        for r in res: w.writerow(r)
    print("CSV:",a.out_csv)

if __name__=="__main__": main()

'''
open('/content/judge_trackB.py','w').write(_J); print('judge written')

judge written


In [ ]:
#@title 2. backbone 선정 + cif->pdb + 정규화(번호충돌 제거) + 충돌검증
import os, glob, gzip, json
import gemmi
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')

WORK='/content/drive/MyDrive/htra1_v3_short'  #@param {type:"string"}
N_PER_RUN=30   #@param {type:"integer"}
TARGET_AC=WORK+'/AC_target.pdb'
BB_ROOT=WORK+'/backbones'
TOPJSON=WORK+'/rerun_combo/top_backbones.json'
assert os.path.isfile(TARGET_AC), 'AC_target.pdb 없음'
assert os.path.isfile(TOPJSON), 'top_backbones.json 없음'
top=json.load(open(TOPJSON))
print('top_backbones.json runs:', list(top.keys()))

def find_cifs(run, n):
    d=f'{BB_ROOT}/{run}'
    if not (isinstance(top, dict) and run in top):
        print(f'  [{run}] top json에 없음'); return []
    cifs=[]
    for fname in top[run][:n]:
        fname=fname if isinstance(fname,str) else (fname.get('name') or fname.get('design'))
        p=os.path.join(d, fname)
        if os.path.isfile(p): cifs.append(p)
        else: print(f'  [경고] 없음: {p}')
    print(f'  [{run}] 선정: {len(cifs)}개')
    return cifs

def cif_to_norm_pdb(cifgz, out_pdb):
    txt=gzip.open(cifgz,'rt').read(); tmp='/content/_bb.cif'; open(tmp,'w').write(txt)
    st=gemmi.read_structure(tmp); pdb=st.make_pdb_string()
    chains={}
    for ln in pdb.splitlines():
        if not ln.startswith(('ATOM','HETATM')): continue
        if ln[16] not in (' ','A'): continue
        chains.setdefault(ln[21],[]).append(ln)
    nres=lambda L: len({l[22:26] for l in L})
    binder=min(chains,key=lambda c:nres(chains[c]))
    order=[binder]+[c for c in chains if c!=binder]; ids=iter('ABCDE'); out=[]; ser=1
    for k,ch in enumerate(order):
        nc=next(ids); rm={}; nx=(1 if k==0 else 1000*k)
        for l in chains[ch]:
            rs=l[22:26]
            if rs not in rm: rm[rs]=nx; nx+=1
            out.append(l[:6]+f'{ser:5d}'+l[11:21]+nc+f'{rm[rs]:4d}'+l[26:]); ser+=1
        out.append('TER\n')
    out.append('END\n'); open(out_pdb,'w').writelines(l if l.endswith('\n') else l+'\n' for l in out)

BB_DIR='/content/bb_norm'; os.makedirs(BB_DIR,exist_ok=True)
chosen=[]
for run in ['L06','L09','L12']:
    for cif in find_cifs(run, N_PER_RUN):
        tag=os.path.basename(cif).replace('.cif.gz','').replace('cfg_','')
        outp=f'{BB_DIR}/{tag}.pdb'; cif_to_norm_pdb(cif,outp); chosen.append((run,tag,outp))

print(f'\n선정 backbone {len(chosen)}개')
bad=0
for run,tag,outp in chosen:
    byc={}
    for ln in open(outp):
        if ln[:4]=='ATOM': byc.setdefault(ln[21],set()).add(int(ln[22:26]))
    ov=byc.get('A',set())&(byc.get('B',set())|byc.get('C',set()))
    if ov: bad+=1; print(f'  [충돌✗] {tag}')
print(f'충돌 있는 backbone: {bad}개 (0이어야 정상)')
BACKBONES=chosen

# bb_norm을 Drive에 백업 (런타임 끊겨도 재폴딩용 backbone 안 날아감)
os.system('cp -r /content/bb_norm /content/drive/MyDrive/htra1_v3_short/bb_norm')
print('bb_norm -> Drive 백업 완료')

top_backbones.json runs: ['L06', 'L09', 'L12']
  [L06] 선정: 30개
  [L09] 선정: 30개
  [L12] 선정: 30개

선정 backbone 90개
충돌 있는 backbone: 0개 (0이어야 정상)
bb_norm -> Drive 백업 완료


In [ ]:
#@title 3. MPNN 8서열/backbone (target 고정) + 고유서열 검증 + Drive 저장
import os, csv
from colabdesign.mpnn import mk_mpnn_model, clear_mem
MANIFEST=WORK+'/diversify_manifest.csv'
rows=[]; allseqs=[]
mp=mk_mpnn_model()
for run,tag,pdb in BACKBONES:
    mp.prep_inputs(pdb_filename=pdb, chain="A,B,C", fix_pos="B,C")   # target(B,C) 고정, binder(A)만 설계
    O=mp.sample(num=8, temperature=0.1)
    seqs=[s.split("/")[0] for s in O["seq"]]                          # binder=첫 chain(/-split)
    for si,seq in enumerate(seqs):
        rows.append({"tag":f"{tag}_s{si}","backbone_pdb":pdb,"s_idx":si,"seq":seq})
        allseqs.append(seq)
    print(f'{tag}: 8서열 예: {seqs[0]}')
with open(MANIFEST,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=["tag","backbone_pdb","s_idx","seq"]); w.writeheader(); w.writerows(rows)
uniq=len(set(allseqs))
print(f'\n총 {len(rows)}서열, 고유 {uniq}  <- 고유수가 총수와 비슷해야 정상(슬라이스 OK). manifest -> Drive')
print('>>> 고유수가 크게 작으면(예: backbone수와 같으면) 슬라이스 버그. 멈추고 알릴 것.')

L06_htra1_v3_L06_130_model_7: 8서열 예: LKPVKTK
L06_htra1_v3_L06_140_model_7: 8서열 예: EVVLP
L06_htra1_v3_L06_64_model_2: 8서열 예: KPKPLI
L06_htra1_v3_L06_22_model_0: 8서열 예: GRTAL
L06_htra1_v3_L06_37_model_2: 8서열 예: PLLLRE
L06_htra1_v3_L06_27_model_2: 8서열 예: VKVRLPE
L06_htra1_v3_L06_43_model_3: 8서열 예: MPNRGLM
L06_htra1_v3_L06_173_model_3: 8서열 예: RKVYP
L06_htra1_v3_L06_4_model_6: 8서열 예: ARPPEL
L06_htra1_v3_L06_31_model_7: 8서열 예: MAILG
L06_htra1_v3_L06_5_model_6: 8서열 예: RVLGG
L06_htra1_v3_L06_26_model_5: 8서열 예: RPVVWG
L06_htra1_v3_L06_162_model_3: 8서열 예: GKVYE
L06_htra1_v3_L06_137_model_2: 8서열 예: SMRIP
L06_htra1_v3_L06_88_model_7: 8서열 예: PKPRV
L06_htra1_v3_L06_154_model_0: 8서열 예: LQVER
L06_htra1_v3_L06_184_model_7: 8서열 예: GPPRI
L06_htra1_v3_L06_79_model_2: 8서열 예: LPEGK
L06_htra1_v3_L06_140_model_2: 8서열 예: GSGLQ
L06_htra1_v3_L06_16_model_2: 8서열 예: EMQEALK
L06_htra1_v3_L06_88_model_5: 8서열 예: SRTLG
L06_htra1_v3_L06_66_model_4: 8서열 예: VLQAPS
L06_htra1_v3_L06_58_model_3: 8서열 예: PGASALG
L06_htra1_v3_

In [ ]:
import os, csv, numpy as np
from colabdesign import mk_afdesign_model, clear_mem
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'
MANIFEST=WORK+'/diversify_manifest.csv'
OUT_DIR=WORK+'/diversify_out'; os.makedirs(OUT_DIR,exist_ok=True)
def com(pdb,chain):
    pts=[(float(l[30:38]),float(l[38:46]),float(l[46:54])) for l in open(pdb) if l[:4]=='ATOM' and l[13:15]=='CA' and l[21]==chain]
    return np.mean(pts,axis=0) if pts else None
def shortest(pdb):
    import collections; c=collections.Counter(l[21] for l in open(pdb) if l[:4]=='ATOM' and l[13:15]=='CA'); return min(c,key=c.get)

man=list(csv.DictReader(open(MANIFEST)))
# 중복 서열 제거 (같은 서열 = 같은 폴딩 결과, CU 아낌)
seen=set(); man=[r for r in man if not (r["seq"] in seen or seen.add(r["seq"]))]
man.sort(key=lambda r:r["backbone_pdb"])
print(f"고유 서열만 {len(man)}개 폴딩")

cur_bb=None; model=None; done=0; total=len(man)
for r in man:
    out=f'{OUT_DIR}/{r["tag"]}.pdb'
    if os.path.isfile(out): print(r["tag"],'skip'); done+=1; continue
    if r["backbone_pdb"]!=cur_bb:
        clear_mem(); cur_bb=r["backbone_pdb"]
        model=mk_afdesign_model(protocol="binder", use_templates=True, data_dir="/content")
        model.prep_inputs(cur_bb, target_chain="B,C", binder_chain="A",
                          rm_target=False, rm_target_seq=False, rm_target_sc=False,
                          use_binder_template=True, rm_template_ic=False)
        ic=com(cur_bb,'A')
    model.predict(seq=r["seq"], num_recycles=3, num_models=1, verbose=False)
    model.save_current_pdb(out)
    oc=com(out,shortest(out)); mov=round(float(np.linalg.norm(ic-oc)),1) if ic is not None and oc is not None else None
    done+=1
    print(f'[{done}/{total}] {r["tag"]} seq {r["seq"]} | 이동 {mov}Å {"OK" if mov is not None and mov<8 else "⚠"}')
print('완료. 저장:',OUT_DIR)

고유 서열만 552개 폴딩
[1/552] L06_htra1_v3_L06_126_model_4_s0 seq MVKAVR | 이동 0.6Å OK
[2/552] L06_htra1_v3_L06_126_model_4_s1 seq LVKSVR | 이동 0.6Å OK
[3/552] L06_htra1_v3_L06_126_model_4_s2 seq MVKSVR | 이동 0.7Å OK
[4/552] L06_htra1_v3_L06_126_model_4_s3 seq MVKPVR | 이동 0.6Å OK
[5/552] L06_htra1_v3_L06_126_model_4_s5 seq SVKAVR | 이동 0.6Å OK
[6/552] L06_htra1_v3_L06_126_model_4_s6 seq GVKSVR | 이동 0.7Å OK
L06_htra1_v3_L06_130_model_7_s0 skip
L06_htra1_v3_L06_130_model_7_s1 skip
L06_htra1_v3_L06_130_model_7_s2 skip
L06_htra1_v3_L06_130_model_7_s3 skip
L06_htra1_v3_L06_130_model_7_s4 skip
L06_htra1_v3_L06_130_model_7_s7 skip
[13/552] L06_htra1_v3_L06_133_model_7_s0 seq KFTTRSE | 이동 0.9Å OK
[14/552] L06_htra1_v3_L06_133_model_7_s2 seq MFTTRAE | 이동 0.9Å OK
[15/552] L06_htra1_v3_L06_133_model_7_s4 seq KFYTRSE | 이동 0.9Å OK
[16/552] L06_htra1_v3_L06_133_model_7_s5 seq SFITREE | 이동 0.6Å OK
[17/552] L06_htra1_v3_L06_133_model_7_s6 seq KFITRSE | 이동 0.9Å OK
[18/552] L06_htra1_v3_L06_137_model_2_s0 seq SMRI

In [ ]:
#@title 5. 판정
import subprocess, sys, pandas as pd, os
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
r=subprocess.run([sys.executable,'/content/clamp_judge.py','--pred_dir',OUT,'--out_csv','/content/clamp.csv'],
                 capture_output=True, text=True)
print("STDOUT:", r.stdout[-500:])
print("STDERR:", r.stderr[-1000:])   # 에러 있으면 여기 뜸
if os.path.isfile('/content/clamp.csv'):
    df=pd.read_csv('/content/clamp.csv')
    real=df[df.real_clamp==True]
    print(f"\nreal_clamp {len(real)}/{len(df)}")
    df['bb']=df.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]
    g=df.groupby('bb').agg(n=('tag','size'),clamps=('real_clamp','sum'),cat_med=('cat_min','median'),nc_med=('ncontact','median'))
    print(g.sort_values('clamps',ascending=False).head(20).to_string())
    print("\n통과 clamp:")
    print(real[['tag','cat_min','ncontact','L3_res','LD_res']].to_string(index=False))
    df.to_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv',index=False)
else:
    print("여전히 csv 없음 -> 위 STDERR 붙여주세요")

STDOUT: real_clamp 408/585  -> /content/clamp.csv

STDERR: 

real_clamp 408/585
                     n  clamps  cat_med  nc_med
bb                                             
L06_L06_140_model_2  8       8    11.75    12.5
L12_L12_59_model_2   8       8    12.80    14.5
L12_L12_59_model_4   8       8    13.70    13.5
L12_L12_54_model_4   8       8    13.55    26.0
L12_L12_50_model_6   8       8    11.35    21.0
L09_L09_9_model_1    8       8    11.45    16.0
L12_L12_173_model_1  8       8    10.45    15.0
L12_L12_36_model_3   8       8    12.40    13.0
L12_L12_99_model_7   8       8     9.10    16.5
L09_L09_39_model_3   8       8    11.40    15.0
L09_L09_95_model_7   8       8    11.10    17.0
L09_L09_152_model_4  8       8    11.00    21.5
L09_L09_102_model_4  8       8    10.25    15.0
L09_L09_101_model_7  7       7     8.80    11.0
L09_L09_135_model_7  7       7    12.10    17.0
L09_L09_79_model_1   7       7    11.00    18.0
L09_L09_152_model_1  8       7     9.60    16.5
L06_L06_

In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
real=df[df.real_clamp==True]
print(f"real_clamp {len(real)}/{len(df)}  ({100*len(real)/len(df):.0f}%)")
print(f"clamp_geom만(촉매침범 포함): {df.clamp_geom.sum()}")
print(f"고유 backbone {df.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0].nunique()}개 중 clamp 낸 backbone: {real.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0].nunique()}개\n")

# clamp 낸 backbone별 통과 수 (상위만)
real2=real.copy()
real2['bb']=real2.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]
top=real2.groupby('bb').agg(clamps=('tag','size'),cat_med=('cat_min','median'),nc_med=('ncontact','median')).sort_values('clamps',ascending=False)
print("clamp 낸 backbone (상위 15):")
print(top.head(15).to_string())

real_clamp 408/585  (70%)
clamp_geom만(촉매침범 포함): 461
고유 backbone 93개 중 clamp 낸 backbone: 75개

clamp 낸 backbone (상위 15):
                     clamps  cat_med  nc_med
bb                                          
L06_L06_140_model_2       8    11.75    12.5
L12_L12_50_model_6        8    11.35    21.0
L12_L12_36_model_3        8    12.40    13.0
L12_L12_99_model_7        8     9.10    16.5
L09_L09_9_model_1         8    11.45    16.0
L12_L12_173_model_1       8    10.45    15.0
L12_L12_59_model_4        8    13.70    13.5
L12_L12_59_model_2        8    12.80    14.5
L09_L09_39_model_3        8    11.40    15.0
L09_L09_152_model_4       8    11.00    21.5
L09_L09_102_model_4       8    10.25    15.0
L12_L12_54_model_4        8    13.55    26.0
L09_L09_95_model_7        8    11.10    17.0
L09_L09_90_model_0        7    12.10    12.0
L12_L12_150_model_2       7    12.90    10.0


In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')

# binder 길이 추출 (L06/L09/L12 -> 대략 6/9/12, 실제는 tag에서)
def blen(tag):
    if 'L06' in tag: return 6
    if 'L09' in tag: return 9
    if 'L12' in tag: return 12
    return None
df['blen']=df.tag.apply(blen)
df['nc_per_res']=df.ncontact/df.blen   # binder 잔기당 target 접촉 수

g=df[df.clamp_geom==True]
print("clamp_geom인 것들의 길이별 ncontact 중앙값:")
for L in [6,9,12]:
    sub=g[g.blen==L]
    if len(sub): print(f"  {L}mer: ncontact 중앙 {sub.ncontact.median():.0f}, nc/res 중앙 {sub.nc_per_res.median():.1f}, n={len(sub)}")
print("\nnc_per_res 분포 (길이 무관 정규화 지표):")
for lo,hi in [(0,1.0),(1.0,1.5),(1.5,2.0),(2.0,2.5),(2.5,9)]:
    n=((g.nc_per_res>=lo)&(g.nc_per_res<hi)).sum()
    print(f"  {lo}-{hi}: {n}개")

clamp_geom인 것들의 길이별 ncontact 중앙값:
  6mer: ncontact 중앙 12, nc/res 중앙 1.9, n=114
  9mer: ncontact 중앙 15, nc/res 중앙 1.7, n=151
  12mer: ncontact 중앙 16, nc/res 중앙 1.3, n=196

nc_per_res 분포 (길이 무관 정규화 지표):
  0-1.0: 23개
  1.0-1.5: 170개
  1.5-2.0: 176개
  2.0-2.5: 76개
  2.5-9: 16개


In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
g=df[df.clamp_geom==True]
print("clamp_geom인 것들의 other(잡접촉) 분포:")
for lo,hi in [(0,1),(2,3),(4,5),(6,8),(9,99)]:
    n=((g.other>=lo)&(g.other<=hi)).sum()
    print(f"  other {lo}-{hi}: {n}개")
print(f"\n길이별 other 중앙값:")
def blen(t): return 6 if 'L06' in t else 9 if 'L09' in t else 12
g=g.copy(); g['blen']=g.tag.apply(blen)
for L in [6,9,12]:
    s=g[g.blen==L]
    if len(s): print(f"  {L}mer: other 중앙 {s.other.median():.0f}")
print(f"\n촉매클리어 + other<=2: {((g.cat_min>8)&(g.other<=2)).sum()}개")
print(f"촉매클리어 + other<=3: {((g.cat_min>8)&(g.other<=3)).sum()}개")

clamp_geom인 것들의 other(잡접촉) 분포:
  other 0-1: 155개
  other 2-3: 150개
  other 4-5: 114개
  other 6-8: 32개
  other 9-99: 10개

길이별 other 중앙값:
  6mer: other 중앙 1
  9mer: other 중앙 2
  12mer: other 중앙 4

촉매클리어 + other<=2: 211개
촉매클리어 + other<=3: 285개


In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
g=df[df.clamp_geom==True].copy()
g['other_frac']=g.other/g.ncontact
def blen(t): return 6 if 'L06' in t else 9 if 'L09' in t else 12
g['blen']=g.tag.apply(blen)
print("길이별 other_frac 중앙값 (이게 길이 무관해야 정규화 성공):")
for L in [6,9,12]:
    s=g[g.blen==L]
    if len(s): print(f"  {L}mer: {s.other_frac.median():.2f}")
print("\nother_frac 분포:")
for lo,hi in [(0,0.15),(0.15,0.25),(0.25,0.35),(0.35,0.5),(0.5,9)]:
    n=((g.other_frac>=lo)&(g.other_frac<hi)).sum()
    print(f"  {lo}-{hi}: {n}개")
print(f"\n촉매클리어 + other_frac<=0.3: {((g.cat_min>8)&(g.other_frac<=0.3)).sum()}개")
print(f"촉매클리어 + other_frac<=0.25: {((g.cat_min>8)&(g.other_frac<=0.25)).sum()}개")

길이별 other_frac 중앙값 (이게 길이 무관해야 정규화 성공):
  6mer: 0.12
  9mer: 0.14
  12mer: 0.23

other_frac 분포:
  0-0.15: 184개
  0.15-0.25: 148개
  0.25-0.35: 93개
  0.35-0.5: 26개
  0.5-9: 10개

촉매클리어 + other_frac<=0.3: 350개
촉매클리어 + other_frac<=0.25: 314개


In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
g=df[(df.clamp_geom==True)&(df.cat_min>8)].copy()
g['other_frac']=g.other/g.ncontact
top=g.sort_values('other_frac').head(15)
print("가장 집중적인 clamp 상위 15 (other_frac 최저 + 촉매회피):")
print(top[['tag','cat_min','ncontact','other','L3_res','LD_res','L3_protomer','LD_protomer']].to_string(index=False))

가장 집중적인 clamp 상위 15 (other_frac 최저 + 촉매회피):
                            tag  cat_min  ncontact  other     L3_res          LD_res  L3_protomer  LD_protomer
L06_htra1_v3_L06_173_model_3_s1     11.2         9      0      [306]      [287, 289]            0            1
L06_htra1_v3_L06_133_model_7_s4     13.6        13      0 [306, 310] [286, 288, 289]            0            1
L06_htra1_v3_L06_133_model_7_s2     13.8        10      0 [305, 306] [286, 288, 289]            0            1
L06_htra1_v3_L06_133_model_7_s0     13.6        11      0      [306] [286, 288, 289]            0            1
 L06_htra1_v3_L06_22_model_0_s1     12.9         7      0 [305, 306]           [287]            0            1
 L06_htra1_v3_L06_22_model_0_s3     13.1         7      0 [305, 306]           [287]            0            1
 L06_htra1_v3_L06_31_model_7_s0     11.3         9      0      [306]      [287, 288]            0            1
 L06_htra1_v3_L06_31_model_7_s6     11.1         9      0      [306]

In [ ]:
import glob, os, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
TARGETS=['L06_htra1_v3_L06_173_model_3_s1','L06_htra1_v3_L06_133_model_7_s4','L06_htra1_v3_L06_133_model_7_s2',
 'L06_htra1_v3_L06_133_model_7_s0','L06_htra1_v3_L06_22_model_0_s1','L06_htra1_v3_L06_22_model_0_s3',
 'L06_htra1_v3_L06_31_model_7_s0','L06_htra1_v3_L06_31_model_7_s6','L06_htra1_v3_L06_31_model_7_s1',
 'L06_htra1_v3_L06_31_model_7_s2','L06_htra1_v3_L06_31_model_7_s3','L06_htra1_v3_L06_31_model_7_s4',
 'L06_htra1_v3_L06_22_model_0_s0','L06_htra1_v3_L06_73_model_1_s6','L06_htra1_v3_L06_88_model_5_s3']
SEG={"LoopA":range(185,209),"L3":range(301,315),"LD":range(285,291)}
def segof(a):
    for s,r in SEG.items():
        if a in r: return s
    return "cat" if a in (220,250,328) else "-"
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        c=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        rn=ln[17:20].strip(); x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        ch.setdefault(c,{}).setdefault(rs,[rs,rn,{}])[2][an]=np.array([x,y,z])
    return {c:[d[k] for k in sorted(d)] for c,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=fa(s,"NAHV");gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None
def bseq(ch,binder):
    return "".join(T2O.get(r[1],'X') for r in ch[binder])

for tgt in TARGETS:
    fs=glob.glob(f'{OUT}/{tgt}.pdb')
    if not fs: print(f'{tgt}: 없음'); continue
    f=fs[0]; ch=parse(f); binder=min(ch,key=lambda c:len(ch[c]))
    ba=np.array([a for r in ch[binder] for a in r[2].values()])
    P=prots([ch[c] for c in ch if c!=binder])
    seq=bseq(ch,binder)
    # protomer별 L3/LD/LoopA 곁사슬 접촉만
    lines=[]
    for pi,p in enumerate(P):
        o=offof(p)
        if o is None: continue
        for r in p:
            auth=r[0]-o; seg=segof(auth)
            if seg not in ("L3","LD","LoopA","cat"): continue
            an=list(r[2].keys()); T=np.array(list(r[2].values()))
            dmin=np.sqrt(((ba[:,None,:]-T[None,:,:])**2).sum(-1)).min(0)
            da=float(dmin.min())
            sc=[i for i,x in enumerate(an) if x not in("N","CA","C","O")]
            ds=float(dmin[sc].min()) if sc else 9e9
            if da<=4.5:
                w="곁" if ds<=4.5 else "백"
                lines.append(f"P{pi}:{seg}{auth}({r[1]}){round(da,1)}[{w}]")
    tag=tgt.replace('htra1_v3_','')
    print(f"{tag:26s} {seq:14s} | " + " ".join(lines))

L06_L06_173_model_3_s1     RPVYP          | P0:L3306(GLU)3.5[곁] P1:LoopA187(GLU)3.6[곁] P1:LoopA200(PRO)2.9[곁] P1:LoopA201(VAL)4.1[백] P1:LoopA202(ALA)3.8[백] P1:LoopA203(SER)3.8[곁] P1:LD286(PHE)4.3[백] P1:LD287(SER)1.7[곁] P1:LD289(GLN)3.6[곁]
L06_L06_133_model_7_s4     KFYTRSE        | P0:L3301(GLN)3.3[백] P0:L3305(LYS)3.0[백] P0:L3306(GLU)3.2[곁] P0:L3307(LEU)3.7[곁] P0:L3308(GLY)3.6[백] P0:L3310(ARG)4.0[곁] P1:LoopA185(HIS)4.3[곁] P1:LoopA187(GLU)4.4[곁] P1:LoopA200(PRO)4.3[곁] P1:LD286(PHE)3.2[곁] P1:LD288(LEU)3.6[곁] P1:LD289(GLN)3.0[곁]
L06_L06_133_model_7_s2     MFTTRAE        | P0:L3305(LYS)2.4[곁] P0:L3306(GLU)3.1[곁] P0:L3307(LEU)3.8[곁] P0:L3308(GLY)3.4[백] P1:LoopA185(HIS)4.2[곁] P1:LoopA187(GLU)4.2[곁] P1:LD286(PHE)3.3[곁] P1:LD288(LEU)3.7[곁] P1:LD289(GLN)3.3[곁]
L06_L06_133_model_7_s0     KFTTRSE        | P0:L3305(LYS)2.3[백] P0:L3306(GLU)3.2[곁] P0:L3307(LEU)3.9[곁] P0:L3308(GLY)3.6[백] P1:LoopA185(HIS)4.2[곁] P1:LoopA187(GLU)4.2[곁] P1:LoopA200(PRO)4.5[곁] P1:LD286(PHE)3.1[곁] P1:LD288(LEU)3.7[곁] P1:LD

In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
g=df[(df.clamp_geom==True)&(df.cat_min>8)&(df.L3_protomer!=df.LD_protomer)].copy()
g['other_frac']=g.other/g.ncontact
def blen(t): return 6 if 'L06' in t else 9 if 'L09' in t else 12
g['blen']=g.tag.apply(blen)
for L in [6,9,12]:
    sub=g[g.blen==L].sort_values('other_frac').head(6)
    print(f"\n=== {L}mer 상위 6 (L3/LD 다른 protomer, 촉매회피, other_frac 최저) ===")
    print(sub[['tag','cat_min','ncontact','other','L3_res','LD_res']].to_string(index=False))


=== 6mer 상위 6 (L3/LD 다른 protomer, 촉매회피, other_frac 최저) ===
                            tag  cat_min  ncontact  other     L3_res          LD_res
L06_htra1_v3_L06_137_model_2_s5     13.9         8      0 [306, 310]      [287, 289]
L06_htra1_v3_L06_133_model_7_s4     13.6        13      0 [306, 310] [286, 288, 289]
L06_htra1_v3_L06_133_model_7_s2     13.8        10      0 [305, 306] [286, 288, 289]
L06_htra1_v3_L06_137_model_2_s1     12.6         6      0 [306, 310] [286, 287, 289]
L06_htra1_v3_L06_133_model_7_s0     13.6        11      0      [306] [286, 288, 289]
L06_htra1_v3_L06_140_model_2_s3     11.7        12      0 [306, 310]      [287, 289]

=== 9mer 상위 6 (L3/LD 다른 protomer, 촉매회피, other_frac 최저) ===
                            tag  cat_min  ncontact  other L3_res          LD_res
L09_htra1_v3_L09_152_model_1_s2      9.5        17      0  [305] [286, 287, 289]
L09_htra1_v3_L09_152_model_1_s1      9.6        16      0  [305] [286, 287, 289]
L09_htra1_v3_L09_152_model_0_s0     11.8  

In [ ]:
import pandas as pd, glob, os
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'
OUT=WORK+'/diversify_out'
df=pd.read_csv(WORK+'/clamp.csv')

# 각 폴딩 PDB에서 binder 서열 읽기
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def bseq(tag):
    p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): return ''
    ch={}
    for ln in open(p):
        if ln[:4]=='ATOM' and ln[16] in (' ','A'):
            ch.setdefault(ln[21],{}).setdefault(int(ln[22:26]),ln[17:20].strip())
    binder=min(ch,key=lambda c:len(ch[c]))
    return ''.join(T2O.get(ch[binder][r],'X') for r in sorted(ch[binder]))

df['seq']=df.tag.apply(bseq)
def blen(t): return 6 if 'L06' in t else 9 if 'L09' in t else 12
df['length']=df.tag.apply(blen)
df['other_frac']=df.other/df.ncontact

# 진짜 clamp = L3/LD 다른 protomer + 촉매회피
clamp=df[(df.clamp_geom==True)&(df.cat_min>8)&(df.L3_protomer!=df.LD_protomer)].copy()
clamp=clamp.sort_values(['length','other_frac'])

# 전체 clamp 상세 CSV
cols=['tag','seq','length','cat_min','ncontact','other','other_frac','L3_res','LD_res','L3_protomer','LD_protomer']
clamp[cols].to_csv(WORK+'/clamp_candidates_20260727.csv', index=False)

# 서열 중복 제거한 후보 리스트 (합성용)
uniq=clamp.drop_duplicates('seq').sort_values(['length','other_frac'])
uniq[cols].to_csv(WORK+'/clamp_candidates_unique_seq_20260727.csv', index=False)

# 요약 텍스트
lines=[]
lines.append(f"HTRA1 inter-protomer clamp 후보 (본run 2026-07-27)")
lines.append(f"판정: L3(305/306) 곁사슬(protomer X) + LD(287/289) 곁사슬(protomer Y!=X) + 촉매>8A")
lines.append(f"전체 폴딩 {len(df)}개 중 clamp {len(clamp)}개, 고유서열 {uniq.seq.nunique()}개\n")
for L in [6,9,12]:
    sub=uniq[uniq.length==L].head(8)
    lines.append(f"=== {L}mer 상위 (other_frac 최저순) ===")
    for _,r in sub.iterrows():
        lines.append(f"  {r.seq:14s} {r.tag.replace('htra1_v3_',''):26s} cat{r.cat_min} nc{r.ncontact} other{r.other} L3{r.L3_res} LD{r.LD_res}")
    lines.append("")
txt="\n".join(lines)
open(WORK+'/clamp_candidates_summary_20260727.txt','w').write(txt)

print(txt)
print("\n저장됨:")
print("  clamp_candidates_20260727.csv (전체 clamp)")
print("  clamp_candidates_unique_seq_20260727.csv (고유서열)")
print("  clamp_candidates_summary_20260727.txt (요약)")

HTRA1 inter-protomer clamp 후보 (본run 2026-07-27)
판정: L3(305/306) 곁사슬(protomer X) + LD(287/289) 곁사슬(protomer Y!=X) + 촉매>8A
전체 폴딩 585개 중 clamp 407개, 고유서열 403개

=== 6mer 상위 (other_frac 최저순) ===
  KFTTRSE        L06_L06_133_model_7_s0     cat13.6 nc11 other0 L3[306] LD[286, 288, 289]
  MFTTRAE        L06_L06_133_model_7_s2     cat13.8 nc10 other0 L3[305, 306] LD[286, 288, 289]
  KFYTRSE        L06_L06_133_model_7_s4     cat13.6 nc13 other0 L3[306, 310] LD[286, 288, 289]
  SMHIP          L06_L06_137_model_2_s1     cat12.6 nc6 other0 L3[306, 310] LD[286, 287, 289]
  SPHLP          L06_L06_137_model_2_s5     cat13.9 nc8 other0 L3[306, 310] LD[287, 289]
  GSGLQ          L06_L06_140_model_2_s0     cat11.8 nc11 other0 L3[306, 310] LD[287, 289]
  GSGLA          L06_L06_140_model_2_s2     cat12.1 nc11 other0 L3[306] LD[287, 289]
  GSGLE          L06_L06_140_model_2_s3     cat11.7 nc12 other0 L3[306, 310] LD[287, 289]

=== 9mer 상위 (other_frac 최저순) ===
  TKTEKVLSLE     L09_L09_152_model_0_s0     cat1

In [ ]:
import glob, os, numpy as np, pandas as pd
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
def binder_plddt(tag):
    p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): return None
    ch={}
    for ln in open(p):
        if ln[:4]=='ATOM' and ln[13:15]=='CA':
            ch.setdefault(ln[21],[]).append(float(ln[60:66]))  # B-factor 열
    if not ch: return None
    binder=min(ch,key=lambda c:len(ch[c]))
    v=np.array(ch[binder])
    return v.mean()   # 0-1 스케일이면 그대로, 0-100이면 /100
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
clamp=df[(df.clamp_geom==True)&(df.cat_min>8)&(df.L3_protomer!=df.LD_protomer)].copy()
clamp['binder_plddt']=clamp.tag.apply(binder_plddt)
print("clamp 후보 binder pLDDT 분포:")
print(clamp['binder_plddt'].describe())
print("\npLDDT 상위 15 clamp:")
print(clamp.sort_values('binder_plddt',ascending=False).head(15)[['tag','binder_plddt','cat_min','ncontact','L3_res','LD_res']].to_string(index=False))

clamp 후보 binder pLDDT 분포:
count    407.000000
mean      68.911433
std       11.264599
min       38.878333
25%       59.869500
50%       71.478000
75%       78.652778
max       87.165556
Name: binder_plddt, dtype: float64

pLDDT 상위 15 clamp:
                            tag  binder_plddt  cat_min  ncontact     L3_res               LD_res
 L09_htra1_v3_L09_35_model_3_s5     87.165556     12.1        17 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_35_model_3_s3     86.768889     12.2        14 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_35_model_3_s2     86.714444     12.0        14 [305, 306] [286, 287, 288, 289]
 L12_htra1_v3_L12_94_model_7_s2     86.265455     11.9        16 [305, 306] [286, 287, 288, 289]
L12_htra1_v3_L12_173_model_1_s4     85.998333     11.3        16 [305, 306]           [287, 289]
 L09_htra1_v3_L09_35_model_3_s4     85.631111     12.6        12 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_35_model_3_s6     85.502222     11.5        16 [305, 306] [286

In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/htra1_v3_short/clamp.csv')
# 아까 pLDDT 계산한 clamp df 재사용하거나 다시 계산
clamp=df[(df.clamp_geom==True)&(df.cat_min>8)&(df.L3_protomer!=df.LD_protomer)].copy()
import glob,os,numpy as np
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
def bp(tag):
    p=f'{OUT}/{tag}.pdb'
    ch={}
    for ln in open(p):
        if ln[:4]=='ATOM' and ln[13:15]=='CA': ch.setdefault(ln[21],[]).append(float(ln[60:66]))
    b=min(ch,key=lambda c:len(ch[c])); return np.mean(ch[b])
clamp['plddt']=clamp.tag.apply(bp)
clamp['other_frac']=clamp.other/clamp.ncontact
def L(t): return 6 if 'L06' in t else 9 if 'L09' in t else 12
clamp['len']=clamp.tag.apply(L)
# 아까 other_frac로 뽑은 6mer 후보들의 pLDDT
print("6mer 집중 clamp(other_frac 최저 8개)의 pLDDT:")
print(clamp[clamp.len==6].sort_values('other_frac').head(8)[['tag','plddt','other_frac','ncontact','cat_min']].to_string(index=False))
print("\n종합 최고 후보 = pLDDT>=80 AND other_frac<=0.3:")
best=clamp[(clamp.plddt>=80)&(clamp.other_frac<=0.3)].sort_values('plddt',ascending=False)
print(f"{len(best)}개")
print(best[['tag','plddt','other_frac','ncontact','cat_min','L3_res','LD_res']].head(20).to_string(index=False))

6mer 집중 clamp(other_frac 최저 8개)의 pLDDT:
                            tag     plddt  other_frac  ncontact  cat_min
L06_htra1_v3_L06_137_model_2_s5 55.568000         0.0         8     13.9
L06_htra1_v3_L06_133_model_7_s4 66.574286         0.0        13     13.6
L06_htra1_v3_L06_133_model_7_s2 67.671429         0.0        10     13.8
L06_htra1_v3_L06_137_model_2_s1 54.214000         0.0         6     12.6
L06_htra1_v3_L06_133_model_7_s0 67.585714         0.0        11     13.6
L06_htra1_v3_L06_140_model_2_s3 58.806000         0.0        12     11.7
L06_htra1_v3_L06_140_model_2_s2 64.366000         0.0        11     12.1
L06_htra1_v3_L06_140_model_7_s6 61.354000         0.0         7     12.3

종합 최고 후보 = pLDDT>=80 AND other_frac<=0.3:
68개
                            tag     plddt  other_frac  ncontact  cat_min     L3_res               LD_res
 L09_htra1_v3_L09_35_model_3_s5 87.165556    0.235294        17     12.1 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_35_model_3_s3 86.768889    0

In [ ]:
#@title 7. clamp 전체 PAE 측정 (shortlist 없이 전부, skip 재개)
import pandas as pd, glob, os, numpy as np, csv
from colabdesign import mk_afdesign_model, clear_mem
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'
OUT=WORK+'/diversify_out'
PAE_OUT=WORK+'/pae_refold'; os.makedirs(PAE_OUT,exist_ok=True)

df=pd.read_csv(WORK+'/clamp.csv')
clamp=df[(df.clamp_geom==True)&(df.cat_min>8)&(df.L3_protomer!=df.LD_protomer)].copy()
print(f"PAE 측정 대상 clamp {len(clamp)}개 (shortlist 없음, 전부)")

man={r['tag']:(r['backbone_pdb'],r['seq']) for r in csv.DictReader(open(WORK+'/diversify_manifest.csv'))}
clamp['bb']=clamp.tag.apply(lambda t: man[t][0] if t in man else '')
clamp=clamp.sort_values('bb')

done=0; total=len(clamp); cur_bb=None; model=None
for _,r in clamp.iterrows():
    tag=r['tag']
    npy=f'{PAE_OUT}/{tag}_ipae.txt'
    if os.path.isfile(npy): done+=1; continue
    if tag not in man: print(tag,'manifest 없음'); continue
    bb,seq=man[tag]
    if bb!=cur_bb:
        clear_mem(); cur_bb=bb
        model=mk_afdesign_model(protocol="binder", use_templates=True, data_dir="/content")
        model.prep_inputs(bb, target_chain="B,C", binder_chain="A",
                          rm_target=False, rm_target_seq=False, rm_target_sc=False,
                          use_binder_template=True, rm_template_ic=False)
    aux=model.predict(seq=seq, num_recycles=3, num_models=1, verbose=False, return_aux=True)
    pae=np.array(aux["pae"]); bl=len(seq)
    ipae_mean=float(pae[:bl, bl:].mean()); ipae_min=float(pae[:bl, bl:].min())
    open(npy,'w').write(f"{ipae_mean:.3f}\t{ipae_min:.3f}\t{bl}")
    done+=1
    if done%20==0 or done<5: print(f"[{done}/{total}] {tag.replace('htra1_v3_','')} ipae_mean {ipae_mean:.1f}")
print(f"\n완료 {done}/{total}. ipae -> {PAE_OUT}/*_ipae.txt")

PAE 측정 대상 clamp 407개 (shortlist 없음, 전부)
L12_htra1_v3_L12_59_model_4_s0 manifest 없음
L06_htra1_v3_L06_59_model_4_s2 manifest 없음
L06_htra1_v3_L06_59_model_4_s3 manifest 없음
L06_htra1_v3_L06_59_model_4_s4 manifest 없음
L06_htra1_v3_L06_59_model_4_s6 manifest 없음
L06_htra1_v3_L06_59_model_4_s7 manifest 없음
L12_htra1_v3_L12_59_model_4_s1 manifest 없음
L06_htra1_v3_L06_59_model_4_s1 manifest 없음
L12_htra1_v3_L12_59_model_4_s7 manifest 없음
L12_htra1_v3_L12_59_model_4_s5 manifest 없음
L12_htra1_v3_L12_59_model_4_s4 manifest 없음
L12_htra1_v3_L12_59_model_4_s3 manifest 없음
L09_htra1_v3_L09_59_model_4_s4 manifest 없음
L09_htra1_v3_L09_59_model_4_s7 manifest 없음
L12_htra1_v3_L12_59_model_4_s2 manifest 없음
L12_htra1_v3_L12_59_model_4_s6 manifest 없음
L06_htra1_v3_L06_59_model_4_s0 manifest 없음
[1/407] L06_L06_126_model_4_s0 ipae_mean 10.5
[2/407] L06_L06_126_model_4_s1 ipae_mean 10.5
[3/407] L06_L06_126_model_4_s6 ipae_mean 10.5
[4/407] L06_L06_126_model_4_s5 ipae_mean 10.4
[20/407] L06_L06_140_model_2_s4 ipae_mean 11.

In [ ]:
#@title 8. PAE triage (길이 편향 확인 + 후보 확정)
import pandas as pd, glob, os, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'
df=pd.read_csv(WORK+'/clamp.csv')
clamp=df[(df.clamp_geom==True)&(df.cat_min>8)&(df.L3_protomer!=df.LD_protomer)].copy()
PAE_OUT=WORK+'/pae_refold'; OUT=WORK+'/diversify_out'
def ipae(tag):
    p=f'{PAE_OUT}/{tag}_ipae.txt'
    if not os.path.isfile(p): return None
    return float(open(p).read().split('\t')[0])
def plddt(tag):
    p=f'{OUT}/{tag}.pdb'; ch={}
    for ln in open(p):
        if ln[:4]=='ATOM' and ln[13:15]=='CA': ch.setdefault(ln[21],[]).append(float(ln[60:66]))
    b=min(ch,key=lambda c:len(ch[c])); return float(np.mean(ch[b]))
clamp['ipae_mean']=clamp.tag.apply(ipae)
clamp['plddt']=clamp.tag.apply(plddt)
clamp['other_frac']=clamp.other/clamp.ncontact
clamp['len']=clamp.tag.apply(lambda t: 6 if 'L06' in t else 9 if 'L09' in t else 12)

print("길이별 ipae_mean 분포 (편향 있나 확인):")
for L in [6,9,12]:
    s=clamp[clamp.len==L]
    print(f"  {L}mer: ipae_mean 중앙 {s.ipae_mean.median():.1f}, pLDDT 중앙 {s.plddt.median():.0f}, n={len(s)}")
print(f"\nipae_mean 분포:")
for lo,hi in [(0,8),(8,10),(10,12),(12,15),(15,99)]:
    print(f"  {lo}-{hi}: {((clamp.ipae_mean>=lo)&(clamp.ipae_mean<hi)).sum()}개")
clamp.to_csv(WORK+'/clamp_with_pae_20260727.csv',index=False)
print("\n저장: clamp_with_pae_20260727.csv")

길이별 ipae_mean 분포 (편향 있나 확인):
  6mer: ipae_mean 중앙 11.5, pLDDT 중앙 63, n=105
  9mer: ipae_mean 중앙 8.2, pLDDT 중앙 72, n=145
  12mer: ipae_mean 중앙 7.4, pLDDT 중앙 75, n=157

ipae_mean 분포:
  0-8: 149개
  8-10: 170개
  10-12: 71개
  12-15: 0개
  15-99: 0개

저장: clamp_with_pae_20260727.csv


In [ ]:
import pandas as pd, csv
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')
man={r['tag']:r['seq'] for r in csv.DictReader(open(WORK+'/diversify_manifest.csv'))}
c['seq']=c.tag.map(man)
c6=c[c.len==6].copy().sort_values('ipae_mean')
print(f"6mer clamp {len(c6)}개 중 iPAE<10: {(c6.ipae_mean<10).sum()}개, iPAE<9: {(c6.ipae_mean<9).sum()}개\n")
print("6mer iPAE 낮은순 상위 20:")
print(c6[['tag','seq','ipae_mean','plddt','other_frac','ncontact','cat_min','L3_res','LD_res']].head(20).to_string(index=False))

6mer clamp 105개 중 iPAE<10: 27개, iPAE<9: 0개

6mer iPAE 낮은순 상위 20:
                            tag     seq  ipae_mean     plddt  other_frac  ncontact  cat_min     L3_res               LD_res
L06_htra1_v3_L06_133_model_7_s0 KFTTRSE      9.517 67.585714    0.000000        11     13.6      [306]      [286, 288, 289]
 L06_htra1_v3_L06_16_model_2_s1 EFQEALL      9.517 74.584286    0.000000        13     11.6      [306]           [287, 289]
 L06_htra1_v3_L06_72_model_0_s3 EIVELES      9.517 77.292857    0.142857        14     13.1      [306]           [287, 289]
 L06_htra1_v3_L06_16_model_2_s0 EMQEALK      9.526 80.555714    0.071429        14     11.2 [305, 306]           [287, 289]
 L06_htra1_v3_L06_16_model_2_s7 EFQKALL      9.536 73.507143    0.000000        14     11.5      [306]           [287, 289]
L06_htra1_v3_L06_133_model_7_s4 KFYTRSE      9.541 66.574286    0.000000        13     13.6 [306, 310]      [286, 288, 289]
 L06_htra1_v3_L06_16_model_2_s2 EFQEALR      9.548 75.475714    0.0

In [ ]:
import pandas as pd, csv, os
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')
# ipae_min 다시 읽기 (txt에 mean\tmin\tbl 저장돼 있음)
PAE=WORK+'/pae_refold'
def imin(tag):
    p=f'{PAE}/{tag}_ipae.txt'
    return float(open(p).read().split('\t')[1]) if os.path.isfile(p) else None
c['ipae_min']=c.tag.apply(imin)
print("길이별 ipae_min 중앙값 (min은 길이 편향 적어야):")
for L in [6,9,12]:
    s=c[c.len==L]
    print(f"  {L}mer: ipae_min 중앙 {s.ipae_min.median():.1f}, n={len(s)}")
print("\nipae_min 분포:")
for lo,hi in [(0,3),(3,5),(5,7),(7,10),(10,99)]:
    print(f"  {lo}-{hi}: {((c.ipae_min>=lo)&(c.ipae_min<hi)).sum()}개")

길이별 ipae_min 중앙값 (min은 길이 편향 적어야):
  6mer: ipae_min 중앙 0.8, n=105
  9mer: ipae_min 중앙 0.8, n=145
  12mer: ipae_min 중앙 0.8, n=157

ipae_min 분포:
  0-3: 390개
  3-5: 0개
  5-7: 0개
  7-10: 0개
  10-99: 0개


In [ ]:
import glob, os, numpy as np, pandas as pd, csv
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
c=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')
man={r['tag']:r['seq'] for r in csv.DictReader(open(WORK+'/diversify_manifest.csv'))}
c['seq']=c.tag.map(man)

SEG_L3=set(range(301,315)); SEG_LD=set(range(285,291))
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        cc=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54]); bf=float(ln[60:66])
        ch.setdefault(cc,{}).setdefault(rs,[rs,ln[17:20].strip(),{},bf])[2][an]=np.array([x,y,z])
    return {k:[d[r] for r in sorted(d)] for k,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=fa(s,"NAHV");gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None

def contact_plddt(tag):
    p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): return None
    ch=parse(p); binder=min(ch,key=lambda k:len(ch[k]))
    brs=ch[binder]                                  # binder 잔기들 [rs,rn,atoms,bfactor]
    ba_by_res={r[0]:np.array(list(r[2].values())) for r in brs}
    P=prots([ch[k] for k in ch if k!=binder])
    # L3/LD target 잔기에 4.5A 이내로 닿는 binder 잔기 찾기
    contact_bres=set()
    for pr in P:
        o=offof(pr)
        if o is None: continue
        for r in pr:
            auth=r[0]-o
            if auth not in SEG_L3 and auth not in SEG_LD: continue
            T=np.array(list(r[2].values()))
            for brs_id,bat in ba_by_res.items():
                if np.sqrt(((bat[:,None,:]-T[None,:,:])**2).sum(-1)).min()<=4.5:
                    contact_bres.add(brs_id)
    if not contact_bres: return None
    # 그 binder 접촉잔기들의 pLDDT(B-factor) 평균
    return float(np.mean([r[3] for r in brs if r[0] in contact_bres]))

c['contact_plddt']=c.tag.apply(contact_plddt)
print("길이별 contact_plddt 중앙값 (접촉잔기만, 길이 편향 적어야):")
for L in [6,9,12]:
    s=c[(c.len==L)&c.contact_plddt.notna()]
    print(f"  {L}mer: 접촉잔기 pLDDT 중앙 {s.contact_plddt.median():.0f}, binder전체 pLDDT 중앙 {s.plddt.median():.0f}, n={len(s)}")
print("\ncontact_plddt 분포:")
for lo,hi in [(0,60),(60,70),(70,80),(80,90),(90,101)]:
    print(f"  {lo}-{hi}: {((c.contact_plddt>=lo)&(c.contact_plddt<hi)).sum()}개")
c.to_csv(WORK+'/clamp_with_pae_20260727.csv',index=False)

길이별 contact_plddt 중앙값 (접촉잔기만, 길이 편향 적어야):
  6mer: 접촉잔기 pLDDT 중앙 65, binder전체 pLDDT 중앙 63, n=105
  9mer: 접촉잔기 pLDDT 중앙 73, binder전체 pLDDT 중앙 72, n=145
  12mer: 접촉잔기 pLDDT 중앙 76, binder전체 pLDDT 중앙 75, n=157

contact_plddt 분포:
  0-60: 80개
  60-70: 98개
  70-80: 134개
  80-90: 95개
  90-101: 0개


In [ ]:
import pandas as pd, csv
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')
# 최종: 접촉잔기 pLDDT>=75 + 촉매회피(이미) + 집중적(other_frac<=0.3) — 길이 컷 없음
best=c[(c.contact_plddt>=75)&(c.other_frac<=0.3)].copy()
best=best.drop_duplicates('seq').sort_values('contact_plddt',ascending=False)
print(f"최종 casein 후보: {len(best)}개 (고유서열, 길이 무관 개별 필터)\n")
print("길이별:", {L:len(best[best.len==L]) for L in [6,9,12]})
print()
print(best[['tag','seq','len','contact_plddt','plddt','other_frac','ncontact','cat_min','L3_res','LD_res']].head(35).to_string(index=False))
best.to_csv(WORK+'/casein_candidates_FINAL_20260727.csv',index=False)
print(f"\n저장: casein_candidates_FINAL_20260727.csv")

최종 casein 후보: 146개 (고유서열, 길이 무관 개별 필터)

길이별: {6: 19, 9: 60, 12: 67}

                            tag           seq  len  contact_plddt     plddt  other_frac  ncontact  cat_min     L3_res               LD_res
 L09_htra1_v3_L09_35_model_3_s5     RPKSRSEVA    9      87.792857 87.165556    0.235294        17     12.1 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_35_model_3_s3     RPKSRSEVK    9      87.495000 86.768889    0.142857        14     12.2 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_35_model_3_s2     RPKSRSEVE    9      87.476667 86.714444    0.142857        14     12.0 [305, 306] [286, 287, 288, 289]
L12_htra1_v3_L12_173_model_1_s4  LYTDEEGKKIPW   12      87.193333 85.998333    0.250000        16     11.3 [305, 306]           [287, 289]
 L12_htra1_v3_L12_94_model_7_s2   AEEVRRLLFER   12      87.066000 86.265455    0.062500        16     11.9 [305, 306] [286, 287, 288, 289]
 L09_htra1_v3_L09_95_model_7_s5    KRKKLTAEDL    9      86.913333 81.006000    0.187500        16

In [ ]:
import pandas as pd, os, shutil
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
c=pd.read_csv(WORK+'/casein_candidates_FINAL_20260727.csv')
c=c[c.seq.notna()].copy()
c['bb']=c.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]

DL=WORK+'/chimerax_check'
if os.path.isdir(DL): shutil.rmtree(DL)
os.makedirs(DL); manifest=[]
for L in [6,9,12]:
    cl=c[c.len==L].sort_values('other_frac')
    # backbone 중복 없이 상위 3개 backbone, 각 그 backbone의 최고 서열 1개
    picked=cl.drop_duplicates('bb').head(3)
    print(f"\n{L}mer — 서로 다른 backbone 3개:")
    for _,r in picked.iterrows():
        src=f'{OUT}/{r.tag}.pdb'
        if not os.path.isfile(src): continue
        fn=f"{L}mer_{r.bb}__{r.seq}__ofrac{r.other_frac:.2f}_cplddt{r.contact_plddt:.0f}_cat{r.cat_min:.0f}.pdb"
        shutil.copy(src, f'{DL}/{fn}')
        manifest.append(f"{fn}\tL3={r.L3_res} LD={r.LD_res} ncontact={r.ncontact}")
        print(f"  {r.bb:22s} {r.seq:14s} ofrac{r.other_frac:.2f} cplddt{r.contact_plddt:.0f} L3{r.L3_res} LD{r.LD_res}")
open(f'{DL}/_INDEX.txt','w').write("\n".join(manifest))
shutil.make_archive(WORK+'/chimerax_check','zip',DL)
print(f"\n총 {len(manifest)}개 PDB -> chimerax_check.zip")


6mer — 서로 다른 backbone 3개:
  L06_L06_16_model_2     EFQEALL        ofrac0.00 cplddt76 L3[306] LD[287, 289]
  L06_L06_73_model_1     VTPGER         ofrac0.00 cplddt80 L3[306] LD[287, 288]
  L06_L06_46_model_1     PPPVKYL        ofrac0.09 cplddt82 L3[305] LD[287]

9mer — 서로 다른 backbone 3개:
  L09_L09_176_model_4    AKLRRELRLP     ofrac0.05 cplddt77 L3[305, 306, 310] LD[286, 287, 288, 289]
  L09_L09_9_model_1      LGFRELKKLE     ofrac0.06 cplddt80 L3[305, 310] LD[286, 287, 288, 289]
  L09_L09_135_model_7    LLERYRAR       ofrac0.06 cplddt79 L3[305, 310] LD[286, 287, 288, 289]

12mer — 서로 다른 backbone 3개:
  L12_L12_64_model_5     SVNQTTLEKLKL   ofrac0.00 cplddt82 L3[305, 306] LD[287, 289]
  L12_L12_94_model_7     SEEVRRLLFEN    ofrac0.06 cplddt86 L3[305, 306] LD[286, 287, 288, 289]
  L12_L12_170_model_2    VEASLRRREAE    ofrac0.06 cplddt82 L3[305, 306, 310] LD[287, 288, 289]

총 9개 PDB -> chimerax_check.zip


In [ ]:
import glob, os, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
DL='/content/drive/MyDrive/htra1_v3_short/chimerax_check'
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def o1(rl): return "".join(T2O.get(r,'X') for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
os.makedirs(DL+'_auth', exist_ok=True)
for f in glob.glob(DL+'/*.pdb'):
    # chain별 잔기 순서·resname 읽기
    ch={}
    for ln in open(f):
        if ln[:4]=='ATOM' and ln[16] in (' ','A'):
            c=ln[21]; rs=int(ln[22:26]); rn=ln[17:20].strip()
            ch.setdefault(c,{}).setdefault(rs,rn)
    binder=min(ch,key=lambda c:len(ch[c]))
    # target chain(들)에서 motif로 offset 구해 auth 부여
    remap={}  # (chain,resseq) -> new_auth
    for c in ch:
        if c==binder:
            for rs in ch[c]: remap[(c,rs)]=rs   # binder는 그대로 1..
            continue
        order=sorted(ch[c]); seq=o1([ch[c][r] for r in order])
        nv=fa(seq,"NAHV"); gg=fa(seq,"GNSGG")
        # protomer별 offset (합쳐진 chain이면 여러 개)
        for k,h in enumerate(nv):
            his_rs=order[h+2]; off=his_rs-220
            # 이 protomer 범위: 이번 NAHV ~ 다음 NAHV 전
            lo=h if k==0 else (gg[k-1]+nv[k])//2
            hi=(gg[k]+nv[k+1])//2 if k+1<len(nv) else len(order)
            for idx in range(lo,hi):
                remap[(c,order[idx])]=order[idx]-off
    # 다시 쓰기 (auth 번호로), protomer는 chain B/C로 분리
    out=[];
    for ln in open(f):
        if ln[:4] not in ('ATOM','TER '):
            out.append(ln); continue
        if ln[:4]=='ATOM':
            c=ln[21]; rs=int(ln[22:26]); key=(c,rs)
            if key in remap:
                na=remap[key]
                out.append(ln[:22]+f'{na:4d}'+ln[26:])
            else: out.append(ln)
        else: out.append(ln)
    open(f'{DL}_auth/{os.path.basename(f)}','w').writelines(out)
print(f"auth 복원 완료 -> {DL}_auth/  (306/287 번호 그대로 뜸)")
import shutil; shutil.make_archive('/content/drive/MyDrive/htra1_v3_short/chimerax_check_auth','zip',DL+'_auth')
print("zip: chimerax_check_auth.zip")

auth 복원 완료 -> /content/drive/MyDrive/htra1_v3_short/chimerax_check_auth/  (306/287 번호 그대로 뜸)
zip: chimerax_check_auth.zip


In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_candidates_FINAL_20260727.csv')
hit=c[c.seq.astype(str).str.contains('VTPGER')]
print(hit[['tag','seq','len','contact_plddt','other_frac','ncontact','cat_min','L3_res','LD_res']].to_string(index=False))

                           tag    seq  len  contact_plddt  other_frac  ncontact  cat_min L3_res     LD_res
L06_htra1_v3_L06_73_model_1_s0 VTPGER    6        80.4525         0.0         9     11.6  [306] [287, 288]


In [ ]:
import pandas as pd, glob, os, numpy as np, csv
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
c=pd.read_csv(WORK+'/casein_candidates_FINAL_20260727.csv')
c=c[c.seq.notna()].copy()

CARBOXYL={'OE1','OE2'}; POS={'NH1','NH2','NE','NZ'}  # Glu306 카복실 / Arg,Lys 양전하
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        cc=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        ch.setdefault(cc,{}).setdefault(rs,[rs,ln[17:20].strip(),{}])[2][an]=np.array([x,y,z])
    return {k:[d[r] for r in sorted(d)] for k,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=fa(s,"NAHV");gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None

def saltbridge(tag):
    p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): return (None,None)
    ch=parse(p); binder=min(ch,key=lambda k:len(ch[k]))
    # binder 양전하 원자들
    pos_atoms=[a for r in ch[binder] for an,a in r[2].items() if an in POS]
    if not pos_atoms: return (False, 99.0)   # binder에 Arg/Lys 없음
    pos=np.array(pos_atoms)
    # 306 카복실 산소 찾기 (모든 protomer)
    P=prots([ch[k] for k in ch if k!=binder]); best=99.0
    for pr in P:
        o=offof(pr)
        if o is None: continue
        r306=next((r for r in pr if r[0]-o==306), None)
        if not r306: continue
        oes=[a for an,a in r306[2].items() if an in CARBOXYL]
        for oe in oes:
            d=float(np.linalg.norm(pos-oe[None],axis=1).min())
            best=min(best,d)
    return (best<=4.5, round(best,1))

res=c.tag.apply(saltbridge)
c['saltbridge']=[r[0] for r in res]
c['sb_dist']=[r[1] for r in res]
sb=c[c.saltbridge==True].drop_duplicates('seq')
nosb=c[c.saltbridge==False].drop_duplicates('seq')
print(f"salt-bridge형: {len(sb)}개 (306 카복실↔binder Arg/Lys <=4.5A)")
print(f"비-salt-bridge형: {len(nosb)}개\n")
print("길이별 salt-bridge형:", {L:len(sb[sb.len==L]) for L in [6,9,12]})
print("\n=== salt-bridge형 상위 (sb_dist 짧은순) ===")
print(sb.sort_values('sb_dist')[['tag','seq','len','sb_dist','contact_plddt','other_frac','cat_min','L3_res','LD_res']].head(30).to_string(index=False))
c.to_csv(WORK+'/casein_candidates_FINAL_20260727.csv',index=False)
sb.sort_values('sb_dist').to_csv(WORK+'/casein_saltbridge_20260727.csv',index=False)
print("\n저장: casein_saltbridge_20260727.csv")

salt-bridge형: 49개 (306 카복실↔binder Arg/Lys <=4.5A)
비-salt-bridge형: 96개

길이별 salt-bridge형: {6: 6, 9: 17, 12: 26}

=== salt-bridge형 상위 (sb_dist 짧은순) ===
                            tag          seq  len  sb_dist  contact_plddt  other_frac  cat_min          L3_res               LD_res
L12_htra1_v3_L12_170_model_2_s1  VEALLRRREEE   12      1.4      76.735000    0.176471     11.9 [305, 306, 310]           [287, 289]
 L09_htra1_v3_L09_24_model_7_s2    EEERRLLRE    9      2.5      78.257500    0.176471     11.5      [305, 306]      [287, 288, 289]
 L12_htra1_v3_L12_80_model_3_s4 AARREERIARRR   12      2.8      77.848333    0.263158     12.5 [305, 306, 310]           [287, 289]
 L12_htra1_v3_L12_80_model_3_s7 AERREERIKRRR   12      2.8      79.442000    0.277778     12.4 [305, 306, 310]           [287, 289]
 L09_htra1_v3_L09_90_model_0_s5     ARVREELR    9      2.9      82.142500    0.142857     12.0      [305, 306]           [287, 289]
 L12_htra1_v3_L12_80_model_3_s5 DARREERIARRR   12      3.0

In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_candidates_FINAL_20260727.csv')
sb=c[c.saltbridge==True].drop_duplicates('seq').copy()
# 서열의 양전하(R+K) 개수, 비율
sb['n_pos']=sb.seq.apply(lambda s: sum(s.count(x) for x in 'RK'))
sb['pos_frac']=sb.n_pos/sb.len
# 정상적 clamp = salt-bridge 있되 양전하 과다 아님 (R+K <= 2, 또는 비율 낮음)
clean=sb[(sb.n_pos<=2)].sort_values('contact_plddt',ascending=False)
print(f"salt-bridge형 중 양전하 R+K<=2 (polyArg 배제): {len(clean)}개\n")
print(clean[['tag','seq','len','n_pos','sb_dist','contact_plddt','other_frac','cat_min','L3_res','LD_res']].head(30).to_string(index=False))
print(f"\n(참고) 양전하 R+K 분포: {sb.n_pos.value_counts().sort_index().to_dict()}")

salt-bridge형 중 양전하 R+K<=2 (polyArg 배제): 16개

                            tag          seq  len  n_pos  sb_dist  contact_plddt  other_frac  cat_min     L3_res               LD_res
 L12_htra1_v3_L12_94_model_7_s6  SEEVRRLLFEN   12      2      4.4      85.590000    0.062500     12.0 [305, 306] [286, 287, 288, 289]
L06_htra1_v3_L06_184_model_7_s0        GPPRI    6      1      4.4      85.430000    0.111111     12.4      [306]           [287, 289]
L06_htra1_v3_L06_184_model_7_s3        SRPRI    6      2      3.8      85.320000    0.250000     12.3      [306]           [287, 289]
L06_htra1_v3_L06_184_model_7_s2        APPRI    6      1      4.3      85.040000    0.111111     12.3      [306]           [287, 289]
 L09_htra1_v3_L09_35_model_3_s0    IPLSRSQVK    9      2      3.8      84.911667    0.266667     12.6 [305, 306] [286, 287, 288, 289]
L06_htra1_v3_L06_184_model_7_s4        SEPRI    6      1      4.1      84.730000    0.111111     12.3      [306]           [287, 289]
 L09_htra1_v3_L09

In [ ]:
import pandas as pd
from collections import Counter
from itertools import groupby
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_candidates_FINAL_20260727.csv')
c=c[c.seq.notna()].drop_duplicates('seq').copy()

c['n_pos']   = c.seq.apply(lambda s: sum(s.count(x) for x in 'RK'))     # 양전하 R+K
c['max_aa']  = c.seq.apply(lambda s: max(Counter(s).values()))          # 최다 단일 AA
c['max_run'] = c.seq.apply(lambda s: max(len(list(g)) for _,g in groupby(s)))  # 최장 연속
# flag 열 (배제 아님, 표시만)
c['flag_polyArg']  = c.n_pos>=3            # 양전하 과다 (비특이 흡착 위험)
c['flag_repeat']   = c.max_aa>=3           # 동일 AA 3개+ (조성 편향)
c['flag_runrepeat']= c.max_run>=3          # 연속 반복 3개+ (SPPS 합성 어려움)
# 종합 등급: salt-bridge + 플래그 없음 = A, salt-bridge + 일부 플래그 = B, 비-SB = C
def grade(r):
    if not r.saltbridge: return 'C_no_saltbridge'
    if r.flag_polyArg or r.flag_repeat or r.flag_runrepeat: return 'B_saltbridge_flagged'
    return 'A_clean_saltbridge'
c['grade']=c.apply(grade,axis=1)

cols=['grade','tag','seq','len','saltbridge','n_pos','max_aa','max_run',
      'flag_polyArg','flag_repeat','flag_runrepeat',
      'sb_dist','contact_plddt','other_frac','cat_min','L3_res','LD_res']
master=c.sort_values(['grade','contact_plddt'],ascending=[True,False])[cols]
master.to_csv(WORK+'/casein_MASTER_flagged_20260727.csv',index=False)

print("등급 분포:", master.grade.value_counts().to_dict())
print("\n=== A등급 (salt-bridge + 플래그 없음) ===")
A=master[master.grade=='A_clean_saltbridge']
print(A[['seq','len','n_pos','sb_dist','contact_plddt','other_frac','cat_min']].to_string(index=False))
print(f"\n저장: casein_MASTER_flagged_20260727.csv (146개 전체, 플래그·등급 포함)")

등급 분포: {'C_no_saltbridge': 96, 'B_saltbridge_flagged': 39, 'A_clean_saltbridge': 10}

=== A등급 (salt-bridge + 플래그 없음) ===
         seq  len  n_pos  sb_dist  contact_plddt  other_frac  cat_min
       GPPRI    6      1      4.4      85.430000    0.111111     12.4
       SRPRI    6      2      3.8      85.320000    0.250000     12.3
       APPRI    6      1      4.3      85.040000    0.111111     12.3
   IPLSRSQVK    9      2      3.8      84.911667    0.266667     12.6
       SEPRI    6      1      4.1      84.730000    0.111111     12.3
  GKVITRPDPA    9      2      3.3      82.612500    0.277778     10.8
       PKPRA    6      2      3.2      79.856667    0.272727     12.2
FARTENGEVVKI   12      2      3.6      79.685000    0.153846     12.4
       PKPRV    6      2      3.1      79.076667    0.230769     12.0
MARTDDGEVIEI   12      1      3.5      78.630000    0.166667     12.6

저장: casein_MASTER_flagged_20260727.csv (146개 전체, 플래그·등급 포함)


In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_school'.replace('school','htra1_v3_short')  # 안전
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_MASTER_flagged_20260727.csv')
c['has_M']=c.seq.str.contains('M')
noM=c[~c.has_M].copy()

print("=== Met 없는 후보, 등급·길이별 상위 ===\n")
for g in ['A_clean_saltbridge','B_saltbridge_flagged']:
    sub=noM[noM.grade==g]
    print(f"[{g}] {len(sub)}개")
    for L in [6,9,12]:
        s=sub[sub.len==L].sort_values('contact_plddt',ascending=False).head(4)
        if len(s):
            print(f"  {L}mer:")
            for _,r in s.iterrows():
                bb=r.tag.replace('htra1_v3_','').rsplit('_',1)[0]
                print(f"    {r.seq:14s} {bb:20s} plddt{r.contact_plddt:.0f} sb{r.sb_dist} ofrac{r.other_frac:.2f} n_pos{r.n_pos} L3{r.L3_res} LD{r.LD_res}")
    print()


=== Met 없는 후보, 등급·길이별 상위 ===

[A_clean_saltbridge] 9개
  6mer:
    GPPRI          L06_L06_184_model_7  plddt85 sb4.4 ofrac0.11 n_pos1 L3[306] LD[287, 289]
    SRPRI          L06_L06_184_model_7  plddt85 sb3.8 ofrac0.25 n_pos2 L3[306] LD[287, 289]
    APPRI          L06_L06_184_model_7  plddt85 sb4.3 ofrac0.11 n_pos1 L3[306] LD[287, 289]
    SEPRI          L06_L06_184_model_7  plddt85 sb4.1 ofrac0.11 n_pos1 L3[306] LD[287, 289]
  9mer:
    IPLSRSQVK      L09_L09_35_model_3   plddt85 sb3.8 ofrac0.27 n_pos2 L3[305, 306] LD[286, 287, 288, 289]
    GKVITRPDPA     L09_L09_79_model_1   plddt83 sb3.3 ofrac0.28 n_pos2 L3[306] LD[287, 288, 289]
  12mer:
    FARTENGEVVKI   L12_L12_3_model_3    plddt80 sb3.6 ofrac0.15 n_pos2 L3[306] LD[287, 289]

[B_saltbridge_flagged] 37개
  9mer:
    RPKSRSEVA      L09_L09_35_model_3   plddt88 sb3.9 ofrac0.24 n_pos3 L3[305, 306] LD[286, 287, 288, 289]
    KRKKLTAEDL     L09_L09_95_model_7   plddt87 sb3.4 ofrac0.19 n_pos4 L3[306, 310] LD[286, 287, 289]
    KRKRLTAE

In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_MASTER_flagged_20260727.csv')  # 146개 (이미 clamp+촉매회피+집중+pLDDT>=75 통과분)
l6=c[c.len==6].copy()
l6['bb']=l6.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]
print(f"146개 최종후보 중 6mer: {len(l6)}개, backbone {l6.bb.nunique()}개\n")
print("6mer backbone별 분포:")
for bb,g in l6.groupby('bb'):
    print(f"\n  {bb}  ({len(g)}개, 등급 {dict(g.grade.value_counts())})")
    for _,r in g.sort_values('contact_plddt',ascending=False).iterrows():
        print(f"    {r.seq:8s} {r.grade[0]} sb{r.saltbridge and r.sb_dist or '-'} plddt{r.contact_plddt:.0f} ofrac{r.other_frac:.2f} n_pos{r.n_pos} maxaa{r.max_aa} L3{r.L3_res} LD{r.LD_res}")

146개 최종후보 중 6mer: 18개, backbone 5개

6mer backbone별 분포:

  L06_L06_16_model_2  (4개, 등급 {'C_no_saltbridge': np.int64(4)})
    EMQEALK  C sb- plddt81 ofrac0.07 n_pos1 maxaa2 L3[305, 306] LD[287, 289]
    EFQEALR  C sb- plddt77 ofrac0.00 n_pos1 maxaa2 L3[305] LD[287, 289]
    EFQEALL  C sb- plddt76 ofrac0.00 n_pos0 maxaa2 L3[306] LD[287, 289]
    EFQKALL  C sb- plddt75 ofrac0.00 n_pos1 maxaa2 L3[306] LD[287, 289]

  L06_L06_184_model_7  (4개, 등급 {'A_clean_saltbridge': np.int64(4)})
    GPPRI    A sb4.4 plddt85 ofrac0.11 n_pos1 maxaa2 L3[306] LD[287, 289]
    SRPRI    A sb3.8 plddt85 ofrac0.25 n_pos2 maxaa2 L3[306] LD[287, 289]
    APPRI    A sb4.3 plddt85 ofrac0.11 n_pos1 maxaa2 L3[306] LD[287, 289]
    SEPRI    A sb4.1 plddt85 ofrac0.11 n_pos1 maxaa1 L3[306] LD[287, 289]

  L06_L06_46_model_1  (4개, 등급 {'C_no_saltbridge': np.int64(4)})
    RPPIEYL  C sb- plddt86 ofrac0.17 n_pos1 maxaa2 L3[305] LD[287]
    RPPVEYL  C sb- plddt84 ofrac0.17 n_pos1 maxaa2 L3[305] LD[287]
    KKPVKYL  C sb- pldd

In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
cl=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')  # clamp_geom+촉매회피 전체 407개
cl6=cl[cl.len==6].copy()
cl6['bb']=cl6.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]
print(f"clamp 전체(필터 전) 6mer: {len(cl6)}개, backbone {cl6.bb.nunique()}개\n")
g=cl6.groupby('bb').agg(n=('tag','size'), plddt_med=('plddt','median'),
                        ofrac_med=('other_frac','median') if 'other_frac' in cl6 else ('cat_min','median'),
                        cat_med=('cat_min','median')).sort_values('n',ascending=False)
print("6mer clamp를 낸 backbone (필터 전):")
print(g.to_string())

clamp 전체(필터 전) 6mer: 105개, backbone 24개

6mer clamp를 낸 backbone (필터 전):
                     n  plddt_med  ofrac_med  cat_med
bb                                                   
L06_L06_140_model_2  8  59.345000   0.038462    11.75
L06_L06_59_model_4   7  73.365000   0.214286    11.80
L06_L06_72_model_0   7  73.685714   0.200000    13.10
L06_L06_126_model_4  6  61.829167   0.176923    10.60
L06_L06_140_model_7  6  63.424000   0.000000    12.40
L06_L06_31_model_7   6  50.185000   0.000000    11.05
L06_L06_162_model_3  6  48.688000   0.458042    15.80
L06_L06_133_model_7  5  66.574286   0.000000    13.60
L06_L06_88_model_5   5  70.006000   0.181818     9.90
L06_L06_16_model_2   5  74.584286   0.000000    11.50
L06_L06_154_model_0  4  41.341000   0.381818    14.95
L06_L06_4_model_6    4  77.545833   0.335664    14.85
L06_L06_46_model_1   4  81.506429   0.166667    10.90
L06_L06_184_model_7  4  79.915000   0.111111    12.30
L06_L06_32_model_1   4  63.637500   0.111111    13.10
L06_L06_13

In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
cl=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')  # 필터 전 clamp 전체
cl['bb']=cl.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]

for L in [6,9,12]:
    s=cl[cl.len==L]
    npass=((s.other_frac<=0.3)&(s.contact_plddt>=75)).sum() if 'contact_plddt' in s else 0
    print(f"\n{'='*60}\n{L}mer: clamp {len(s)}개, backbone {s.bb.nunique()}개, "
          f"pLDDT중앙 {s.contact_plddt.median():.0f}, 필터통과 {npass}개")
    g=s.groupby('bb').agg(n=('tag','size'), plddt_med=('contact_plddt','median'),
                          ofrac_med=('other_frac','median'), cat_med=('cat_min','median'))
    # 필터 통과분(pLDDT>=75 & ofrac<=0.3)이 있는 backbone만, pLDDT순
    g=g[g.plddt_med>=70].sort_values('plddt_med',ascending=False)
    print(f"  pLDDT중앙>=70인 backbone (필터 통과 후보):")
    print(g.head(12).to_string())


6mer: clamp 105개, backbone 24개, pLDDT중앙 65, 필터통과 21개
  pLDDT중앙>=70인 backbone (필터 통과 후보):
                     n  plddt_med  ofrac_med  cat_med
bb                                                   
L06_L06_184_model_7  4  85.180000   0.111111    12.30
L06_L06_46_model_1   4  83.567500   0.166667    10.90
L06_L06_73_model_1   3  80.452500   0.000000    11.80
L06_L06_88_model_7   3  79.076667   0.230769    12.20
L06_L06_4_model_6    4  77.685000   0.335664    14.85
L06_L06_16_model_2   5  76.202500   0.000000    11.50
L06_L06_59_model_4   7  74.750000   0.214286    11.80
L06_L06_88_model_5   5  70.006000   0.181818     9.90

9mer: clamp 145개, backbone 27개, pLDDT중앙 73, 필터통과 60개
  pLDDT중앙>=70인 backbone (필터 통과 후보):
                     n  plddt_med  ofrac_med  cat_med
bb                                                   
L09_L09_35_model_3   6  86.817500   0.200980    12.15
L09_L09_26_model_1   5  82.240000   0.133333    11.80
L09_L09_95_model_7   8  81.490000   0.235294    11.10
L09_L09_90

In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_MASTER_flagged_20260727.csv')
c['bb']=c.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]
c=c[~c.seq.str.contains('M',na=False)]  # Met 제외
# 각 backbone의 대표(pLDDT 최고) 1서열, 길이별로 pLDDT+집중도 상위 backbone
for L in [6,9,12]:
    s=c[c.len==L].copy()
    rep=s.sort_values('contact_plddt',ascending=False).groupby('bb').head(1)
    rep=rep.sort_values(['other_frac']).head(8)  # 집중적인 순
    print(f"\n{L}mer 대표 (backbone별 최고서열, 집중도순, Met제외):")
    for _,r in rep.iterrows():
        print(f"  {r.seq:14s} {r.bb:20s} {r.grade[0]} plddt{r.contact_plddt:.0f} ofrac{r.other_frac:.2f} sb{r.saltbridge and r.sb_dist or '-'} L3{r.L3_res} LD{r.LD_res}")


6mer 대표 (backbone별 최고서열, 집중도순, Met제외):
  EFQEALR        L06_L06_16_model_2   C plddt77 ofrac0.00 sb- L3[305] LD[287, 289]
  VSPGER         L06_L06_73_model_1   C plddt84 ofrac0.10 sb- L3[306] LD[287, 288]
  GPPRI          L06_L06_184_model_7  A plddt85 ofrac0.11 sb4.4 L3[306] LD[287, 289]
  RPPIEYL        L06_L06_46_model_1   C plddt86 ofrac0.17 sb- L3[305] LD[287]
  PKPRA          L06_L06_88_model_7   A plddt80 ofrac0.27 sb3.2 L3[305, 306] LD[287, 289]

9mer 대표 (backbone별 최고서열, 집중도순, Met제외):
  AKLRRELRLP     L09_L09_176_model_4  B plddt77 ofrac0.05 sb4.2 L3[305, 306, 310] LD[286, 287, 288, 289]
  LGFRELKKLE     L09_L09_9_model_1    C plddt80 ofrac0.06 sb- L3[305, 310] LD[286, 287, 288, 289]
  LLERYKAR       L09_L09_135_model_7  C plddt80 ofrac0.06 sb- L3[305, 310] LD[286, 287, 288, 289]
  SFVPLTKLEE     L09_L09_102_model_4  C plddt77 ofrac0.07 sb- L3[305, 306, 310] LD[286, 287, 289]
  EEELEEYRT      L09_L09_26_model_1   C plddt83 ofrac0.07 sb- L3[310] LD[287]
  LPALLLRAG      L09_L09

In [ ]:
import pandas as pd
WORK='/content/drive/MyDrive/htra1_v3_short'
c=pd.read_csv(WORK+'/casein_MASTER_flagged_20260727.csv')
cl=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')
cl['bb']=cl.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]
c['bb']=c.tag.str.replace('htra1_v3_','').str.rsplit('_',n=1).str[0]

# 1) 길이별 backbone 분포 (필터 전 clamp 전체)
rows=[]
for L in [6,9,12]:
    s=cl[cl.len==L]
    for bb,g in s.groupby('bb'):
        rows.append({'length':L,'backbone':bb,'n_clamp':len(g),
                     'plddt_med':round(g.contact_plddt.median(),1),
                     'ofrac_med':round(g.other_frac.median(),3),
                     'cat_med':round(g.cat_min.median(),1),
                     'passed_filter':int(((g.contact_plddt>=75)&(g.other_frac<=0.3)).sum())})
bbdist=pd.DataFrame(rows).sort_values(['length','plddt_med'],ascending=[True,False])
bbdist.to_csv(WORK+'/analysis_backbone_distribution_20260727.csv',index=False)

# 2) 최종 합성 리스트 8개
FINAL_SEQS=['RPPIEYL','GPPRI','LGFRELKKLE','IPLSRSQVK','AKVREELR','GVNLSTLERLKF','AEEVRRLLFER','PTRVYEIDGKIYT']
final=c[c.seq.isin(FINAL_SEQS)].drop_duplicates('seq').copy()
# 화학 분류 라벨 추가
def chem(r):
    if r.saltbridge: return 'salt-bridge'
    e=r.seq.count('E')+r.seq.count('D')
    if e>=3: return 'acidic-core'
    return 'hydrophobic/Pro'
final['binding_chem']=final.apply(chem,axis=1)
final['priority']=final.seq.map({s:i+1 for i,s in enumerate(FINAL_SEQS)})
cols=['priority','seq','len','bb','grade','binding_chem','saltbridge','sb_dist',
      'n_pos','max_aa','contact_plddt','other_frac','cat_min','L3_res','LD_res']
final=final.sort_values('priority')[cols]
final.to_csv(WORK+'/casein_SYNTHESIS_LIST_FINAL_20260727.csv',index=False)

# 3) 요약 텍스트
txt=[]
txt.append("HTRA1 casein 합성 최종 리스트 (2026-07-27)")
txt.append("판정: inter-protomer clamp (L3 곁사슬 protomer X + LD 곁사슬 protomer Y!=X) + 촉매회피(>8A)")
txt.append("      + 집중적(other_frac<=0.3) + 접촉잔기 pLDDT>=75, Met 제외, 화학/backbone 다양성")
txt.append(f"필터 전 clamp: 6mer 105(24bb) / 9mer 145(27bb) / 12mer 157(24bb) — clamp는 길이무관 흔함, pLDDT로 선별")
txt.append("\n=== 합성 우선순위 8개 ===")
for _,r in final.iterrows():
    txt.append(f"{r.priority}. {r.seq:14s} ({r.len}mer, {r.bb}) {r.binding_chem:16s} pLDDT{r.contact_plddt:.0f} ofrac{r.other_frac:.2f} cat{r.cat_min:.0f}  L3{r.L3_res} LD{r.LD_res}")
txt.append("\nCAVEAT: clamp != activator. L3-LD 고정이 활성화/억제 중 어느 방향인지는 casein이 결정.")
txt.append("전체 후보 146개는 casein_MASTER_flagged_20260727.csv 참조.")
open(WORK+'/casein_SYNTHESIS_LIST_FINAL_20260727.txt','w').write("\n".join(txt))

print("\n".join(txt))
print("\n\n=== 저장된 파일 (Drive/htra1_v3_short/) ===")
print("  casein_SYNTHESIS_LIST_FINAL_20260727.csv  (최종 8개 + 화학분류)")
print("  casein_SYNTHESIS_LIST_FINAL_20260727.txt  (요약)")
print("  analysis_backbone_distribution_20260727.csv  (길이별 backbone 전체 분포)")
print("  [기존] casein_MASTER_flagged_20260727.csv  (146개 전체 + 플래그/등급)")
print("  [기존] clamp_with_pae_20260727.csv  (clamp 407개 + pLDDT/PAE)")

HTRA1 casein 합성 최종 리스트 (2026-07-27)
판정: inter-protomer clamp (L3 곁사슬 protomer X + LD 곁사슬 protomer Y!=X) + 촉매회피(>8A)
      + 집중적(other_frac<=0.3) + 접촉잔기 pLDDT>=75, Met 제외, 화학/backbone 다양성
필터 전 clamp: 6mer 105(24bb) / 9mer 145(27bb) / 12mer 157(24bb) — clamp는 길이무관 흔함, pLDDT로 선별

=== 합성 우선순위 8개 ===
1. RPPIEYL        (6mer, L06_L06_46_model_1) hydrophobic/Pro  pLDDT86 ofrac0.17 cat11  L3[305] LD[287]
2. GPPRI          (6mer, L06_L06_184_model_7) salt-bridge      pLDDT85 ofrac0.11 cat12  L3[306] LD[287, 289]
3. LGFRELKKLE     (9mer, L09_L09_9_model_1) hydrophobic/Pro  pLDDT80 ofrac0.06 cat11  L3[305, 310] LD[286, 287, 288, 289]
4. IPLSRSQVK      (9mer, L09_L09_35_model_3) salt-bridge      pLDDT85 ofrac0.27 cat13  L3[305, 306] LD[286, 287, 288, 289]
5. AKVREELR       (9mer, L09_L09_90_model_0) salt-bridge      pLDDT83 ofrac0.14 cat12  L3[305, 306] LD[287, 289]
6. GVNLSTLERLKF   (12mer, L12_L12_64_model_5) hydrophobic/Pro  pLDDT86 ofrac0.06 cat11  L3[305, 306] LD[287, 289]
7. AEEVRRLLFER    (

In [ ]:
import pandas as pd, os, shutil, numpy as np, glob
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
final=pd.read_csv(WORK+'/casein_SYNTHESIS_LIST_FINAL_20260727.csv')
man={r['seq']:r['tag'] for r in __import__('csv').DictReader(open(WORK+'/casein_MASTER_flagged_20260727.csv'))}

T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def o1(rl): return "".join(T2O.get(r,'X') for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o

def auth_restore(src, dst):
    # target 두 protomer가 합쳐진 chain을 auth 번호로 복원 + chain 분리
    ch={}
    for ln in open(src):
        if ln[:4]=='ATOM' and ln[16] in (' ','A'):
            c=ln[21]; rs=int(ln[22:26]); rn=ln[17:20].strip()
            ch.setdefault(c,{}).setdefault(rs,rn)
    binder=min(ch,key=lambda c:len(ch[c]))
    remap={}  # (chain,resseq)->(newchain,newauth)
    for c in ch:
        if c==binder:
            for rs in ch[c]: remap[(c,rs)]=('A',rs)   # binder chain A, 1..
            continue
        order=sorted(ch[c]); seq=o1([ch[c][r] for r in order])
        nv=fa(seq,"NAHV"); gg=fa(seq,"GNSGG")
        prot_chain=iter('BCDE')
        for k,h in enumerate(nv):
            off=order[h+2]-220; nc=next(prot_chain)
            lo=h if k==0 else (gg[k-1]+nv[k])//2
            hi=(gg[k]+nv[k+1])//2 if k+1<len(nv) else len(order)
            for idx in range(lo,hi):
                remap[(c,order[idx])]=(nc, order[idx]-off)
    out=[]
    for ln in open(src):
        if ln[:4]=='ATOM':
            key=(ln[21],int(ln[22:26]))
            if key in remap:
                nc,na=remap[key]; out.append(ln[:21]+nc+f'{na:4d}'+ln[26:])
            else: out.append(ln)
        elif ln[:3]=='TER': out.append(ln)
    out.append('END\n'); open(dst,'w').writelines(out)

DL=WORK+'/synthesis_pdb';
if os.path.isdir(DL): shutil.rmtree(DL)
os.makedirs(DL)
done=[]
for _,r in final.iterrows():
    tag=man.get(r.seq)
    if not tag: print(f"{r.seq}: tag 못찾음"); continue
    src=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(src): print(f"{r.seq}: PDB 없음"); continue
    fn=f"p{r.priority}_{r.seq}_{r.len}mer_{r.bb}_authnum.pdb"
    auth_restore(src, f'{DL}/{fn}')
    done.append(fn)
    print(f"  p{r.priority} {r.seq:14s} -> {fn}")

# 인덱스 + zip
idx="\n".join(f"{r.priority}. {r.seq} ({r.len}mer, {r.bb}, {r.binding_chem}) L3={r.L3_res} LD={r.LD_res}"
              for _,r in final.iterrows())
open(f'{DL}/_INDEX.txt','w').write("우선순위 서열 (auth 번호 복원됨: 306/287/220 그대로 뜸)\n\n"+idx)
shutil.make_archive(WORK+'/synthesis_pdb','zip',DL)
print(f"\n{len(done)}개 PDB (auth 복원) -> synthesis_pdb.zip")
print("chain A=binder, B/C=target protomer, 잔기번호는 3NZI auth")

  p1 RPPIEYL        -> p1_RPPIEYL_6mer_L06_L06_46_model_1_authnum.pdb
  p2 GPPRI          -> p2_GPPRI_6mer_L06_L06_184_model_7_authnum.pdb
  p3 LGFRELKKLE     -> p3_LGFRELKKLE_9mer_L09_L09_9_model_1_authnum.pdb
  p4 IPLSRSQVK      -> p4_IPLSRSQVK_9mer_L09_L09_35_model_3_authnum.pdb
  p5 AKVREELR       -> p5_AKVREELR_9mer_L09_L09_90_model_0_authnum.pdb
  p6 GVNLSTLERLKF   -> p6_GVNLSTLERLKF_12mer_L12_L12_64_model_5_authnum.pdb
  p7 AEEVRRLLFER    -> p7_AEEVRRLLFER_12mer_L12_L12_94_model_7_authnum.pdb
  p8 PTRVYEIDGKIYT  -> p8_PTRVYEIDGKIYT_12mer_L12_L12_39_model_0_authnum.pdb

8개 PDB (auth 복원) -> synthesis_pdb.zip
chain A=binder, B/C=target protomer, 잔기번호는 3NZI auth


In [ ]:
import pandas as pd, os, shutil, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
final=pd.read_csv(WORK+'/casein_SYNTHESIS_LIST_FINAL_20260727.csv')
import csv as _csv
man={r['seq']:r['tag'] for r in _csv.DictReader(open(WORK+'/casein_MASTER_flagged_20260727.csv'))}
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def o1(rl): return "".join(T2O.get(r,'X') for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o

def rebuild(src, dst):
    # 원본에서 각 chain 읽기
    ch={}
    for ln in open(src):
        if ln[:4]=='ATOM' and ln[16] in (' ','A'):
            c=ln[21]; rs=int(ln[22:26])
            ch.setdefault(c,{}).setdefault(rs,[]).append(ln)
    # binder = 가장 짧은 chain
    binder=min(ch,key=lambda c:len(ch[c]))
    out=[]
    # binder -> chain Z, 1..
    for i,rs in enumerate(sorted(ch[binder]),1):
        for ln in ch[binder][rs]:
            out.append(ln[:21]+'Z'+f'{i:4d}'+ln[26:])
    out.append('TER\n')
    # target chain(들) -> motif로 protomer 나누고 auth 번호, chain A/B
    pc=iter('AB')
    for c in ch:
        if c==binder: continue
        order=sorted(ch[c]); seq=o1([T2O_inv(ch[c][r][0]) for r in order]) if False else o1([ch[c][r][0][17:20].strip() for r in order])
        nv=fa(seq,"NAHV"); gg=fa(seq,"GNSGG")
        for k,h in enumerate(nv):
            off=order[h+2]-220; nch=next(pc)
            lo=h if k==0 else (gg[k-1]+nv[k])//2
            hi=(gg[k]+nv[k+1])//2 if k+1<len(nv) else len(order)
            for idx in range(lo,hi):
                rs=order[idx]; auth=rs-off
                for ln in ch[c][rs]:
                    out.append(ln[:21]+nch+f'{auth:4d}'+ln[26:])
            out.append('TER\n')
    out.append('END\n'); open(dst,'w').writelines(out)

DL=WORK+'/synthesis_pdb2'
if os.path.isdir(DL): shutil.rmtree(DL)
os.makedirs(DL)
for _,r in final.iterrows():
    tag=man.get(r.seq)
    if not tag: continue
    src=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(src): continue
    fn=f"p{r.priority}_{r.seq}_{r.len}mer_{r.bb}.pdb"
    rebuild(src, f'{DL}/{fn}')
    print(f"p{r.priority} {r.seq} -> {fn}")
shutil.make_archive(WORK+'/synthesis_pdb2','zip',DL)
print("\n-> synthesis_pdb2.zip : binder=chain Z, target=chain A/B(auth번호)")

p1 RPPIEYL -> p1_RPPIEYL_6mer_L06_L06_46_model_1.pdb
p2 GPPRI -> p2_GPPRI_6mer_L06_L06_184_model_7.pdb
p3 LGFRELKKLE -> p3_LGFRELKKLE_9mer_L09_L09_9_model_1.pdb
p4 IPLSRSQVK -> p4_IPLSRSQVK_9mer_L09_L09_35_model_3.pdb
p5 AKVREELR -> p5_AKVREELR_9mer_L09_L09_90_model_0.pdb
p6 GVNLSTLERLKF -> p6_GVNLSTLERLKF_12mer_L12_L12_64_model_5.pdb
p7 AEEVRRLLFER -> p7_AEEVRRLLFER_12mer_L12_L12_94_model_7.pdb
p8 PTRVYEIDGKIYT -> p8_PTRVYEIDGKIYT_12mer_L12_L12_39_model_0.pdb

-> synthesis_pdb2.zip : binder=chain Z, target=chain A/B(auth번호)


In [ ]:
import pandas as pd, os, numpy as np, csv
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
final=pd.read_csv(WORK+'/casein_SYNTHESIS_LIST_FINAL_20260727.csv')
man={r['seq']:r['tag'] for r in csv.DictReader(open(WORK+'/casein_MASTER_flagged_20260727.csv'))}
SEG={"LoopA":set(range(185,209)),"L3":set(range(301,315)),"LD":set(range(285,291))}
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if ln[:4]!='ATOM' or ln[16] not in (' ','A'): continue
        c=ln[21]; rs=int(ln[22:26]); an=ln[12:16].strip()
        if an.startswith('H'): continue
        ch.setdefault(c,{}).setdefault(rs,[rs,ln[17:20].strip(),{}])[2][an]=np.array([float(ln[30:38]),float(ln[38:46]),float(ln[46:54])])
    return {k:[d[r] for r in sorted(d)] for k,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],'X') for r in rl)
def faa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=faa(s,"NAHV");gg=faa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None
print(f"{'seq':14s} LoopA L3 LD  판정")
for _,r in final.iterrows():
    tag=man.get(r.seq); p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): continue
    ch=parse(p); binder=min(ch,key=lambda c:len(ch[c]))
    ba=np.array([a for rr in ch[binder] for a in rr[2].values()])
    cnt={"LoopA":0,"L3":0,"LD":0}
    for pr in prots([ch[c] for c in ch if c!=binder]):
        o=offof(pr)
        if o is None: continue
        for rr in pr:
            auth=rr[0]-o
            for seg,rng in SEG.items():
                if auth in rng:
                    d=min(float(np.linalg.norm(ba-a[None],axis=1).min()) for a in rr[2].values())
                    if d<=4.5: cnt[seg]+=1
    verdict = "LoopA우세" if cnt["LoopA"]>cnt["L3"]+cnt["LD"] else "junction우세"
    print(f"{r.seq:14s} {cnt['LoopA']:5d}{cnt['L3']:3d}{cnt['LD']:3d}  {verdict}")

seq            LoopA L3 LD  판정
RPPIEYL            7  2  1  LoopA우세
GPPRI              4  1  3  junction우세
LGFRELKKLE         7  5  4  junction우세
IPLSRSQVK          3  4  4  junction우세
AKVREELR           6  3  3  junction우세
GVNLSTLERLKF       8  5  3  junction우세
AEEVRRLLFER        7  4  4  junction우세
PTRVYEIDGKIYT      6  6  3  junction우세


In [ ]:
import pandas as pd, os, numpy as np, csv
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
c=pd.read_csv(WORK+'/clamp_with_pae_20260727.csv')
SEG={"LoopA":set(range(185,209)),"L3":set(range(301,315)),"LD":set(range(285,291)),"Loop2":set(range(345,353))}
CAT={220,250,328}
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if ln[:4]!='ATOM' or ln[16] not in (' ','A'): continue
        c_=ln[21]; rs=int(ln[22:26]); an=ln[12:16].strip()
        if an.startswith('H'): continue
        ch.setdefault(c_,{}).setdefault(rs,[rs,ln[17:20].strip(),{}])[2][an]=np.array([float(ln[30:38]),float(ln[38:46]),float(ln[46:54])])
    return {k:[d[r] for r in sorted(d)] for k,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],'X') for r in rl)
def faa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=faa(s,"NAHV");gg=faa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None
man={r['tag']:r['tag'] for r in csv.DictReader(open(WORK+'/clamp_with_pae_20260727.csv'))}

sites=[]
for tag in c.tag:
    p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): sites.append('?'); continue
    ch=parse(p); binder=min(ch,key=lambda k:len(ch[k]))
    ba=np.array([a for rr in ch[binder] for a in rr[2].values()])
    tally={k:0 for k in SEG}; tally['cat']=0; tally['other']=0
    for pr in prots([ch[k] for k in ch if k!=binder]):
        o=offof(pr)
        if o is None: continue
        for rr in pr:
            auth=rr[0]-o; d=min(float(np.linalg.norm(ba-a[None],axis=1).min()) for a in rr[2].values())
            if d>4.5: continue
            placed=False
            for seg,rng in SEG.items():
                if auth in rng: tally[seg]+=1; placed=True; break
            if auth in CAT: tally['cat']+=1; placed=True
            if not placed: tally['other']+=1
    sites.append(max(tally,key=tally.get))
c['site']=sites
print("전체 clamp의 dominant site 분포:")
print(c.site.value_counts().to_string())
print("\n길이별:")
print(c.groupby(['len','site']).size().unstack(fill_value=0).to_string())
c.to_csv(WORK+'/clamp_with_site_20260727.csv',index=False)
print("\n저장: clamp_with_site_20260727.csv")

전체 clamp의 dominant site 분포:
site
LoopA    289
L3        82
other     20
LD        16

길이별:
site  L3  LD  LoopA  other
len                       
6     23   3     73      6
9     24   4    111      6
12    35   9    105      8

저장: clamp_with_site_20260727.csv


In [ ]:
import pandas as pd, os, numpy as np, csv
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
c=pd.read_csv(WORK+'/clamp_with_site_20260727.csv')
man={r['tag']:r['seq'] for r in csv.DictReader(open(WORK+'/casein_MASTER_flagged_20260727.csv'))}
c['seq']=c.tag.map(man)

# 순수 junction = site가 L3 또는 LD (LoopA 아님)
jn=c[c.site.isin(['L3','LD'])].copy()
print(f"순수 junction(site=L3/LD): {len(jn)}개 / 전체 clamp {len(c)}")
print(f"  길이별: {jn.groupby('len').size().to_dict()}")
print(f"  신뢰도(contact_plddt>=75): {(jn.contact_plddt>=75).sum()}개\n")

# LoopA 오염 정도를 직접 재서 확인 (LoopA 접촉 수)
SEG={"LoopA":set(range(185,209)),"L3":set(range(301,315)),"LD":set(range(285,291))}
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if ln[:4]!='ATOM' or ln[16] not in (' ','A'): continue
        c_=ln[21]; rs=int(ln[22:26]); an=ln[12:16].strip()
        if an.startswith('H'): continue
        ch.setdefault(c_,{}).setdefault(rs,[rs,ln[17:20].strip(),{}])[2][an]=np.array([float(ln[30:38]),float(ln[38:46]),float(ln[46:54])])
    return {k:[d[r] for r in sorted(d)] for k,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],'X') for r in rl)
def faa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=faa(s,"NAHV");gg=faa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None
def loopA_count(tag):
    p=f'{OUT}/{tag}.pdb'
    if not os.path.isfile(p): return None
    ch=parse(p); binder=min(ch,key=lambda k:len(ch[k]))
    ba=np.array([a for rr in ch[binder] for a in rr[2].values()])
    n=0
    for pr in prots([ch[k] for k in ch if k!=binder]):
        o=offof(pr)
        if o is None: continue
        for rr in pr:
            if (rr[0]-o) in SEG["LoopA"]:
                if min(float(np.linalg.norm(ba-a[None],axis=1).min()) for a in rr[2].values())<=4.5: n+=1
    return n
jn['loopA_n']=jn.tag.apply(loopA_count)

# 진짜 순수 junction = site L3/LD + LoopA 접촉 적음(<=2) + 신뢰도 + 촉매회피
pure=jn[(jn.contact_plddt>=75)&(jn.loopA_n<=2)&(jn.cat_min>8)].drop_duplicates('seq').sort_values('contact_plddt',ascending=False)
print(f"진짜 순수 junction (site L3/LD + LoopA접촉<=2 + pLDDT>=75 + 촉매회피): {len(pure)}개\n")
print(pure[['seq','len','site','loopA_n','contact_plddt','other_frac','cat_min','L3_res','LD_res']].head(30).to_string(index=False))
pure.to_csv(WORK+'/pure_junction_20260727.csv',index=False)
print(f"\n저장: pure_junction_20260727.csv")

순수 junction(site=L3/LD): 98개 / 전체 clamp 407
  길이별: {6: 26, 9: 28, 12: 44}
  신뢰도(contact_plddt>=75): 29개

진짜 순수 junction (site L3/LD + LoopA접촉<=2 + pLDDT>=75 + 촉매회피): 1개

seq  len site  loopA_n  contact_plddt  other_frac  cat_min     L3_res     LD_res
NaN   12   L3        2          81.13    0.315789     12.7 [305, 306] [287, 289]

저장: pure_junction_20260727.csv


In [ ]:
import pandas as pd, os, shutil, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
pure=pd.read_csv(WORK+'/pure_junction_20260727.csv')
print("순수 junction 1개의 tag/전체정보:")
print(pure[['tag','seq','len','site','loopA_n','contact_plddt','other_frac','cat_min','L3_res','LD_res']].to_string(index=False))
tag=pure.iloc[0]['tag']
print(f"\ntag = {tag}")

# 서열 직접 읽기 (마스터 CSV에 없어서 NaN이었을 수 있음)
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
p=f'{OUT}/{tag}.pdb'
ch={}
for ln in open(p):
    if ln[:4]=='ATOM' and ln[16] in (' ','A'):
        c=ln[21]; rs=int(ln[22:26])
        ch.setdefault(c,{}).setdefault(rs,ln[17:20].strip())
binder=min(ch,key=lambda c:len(ch[c]))
seq=''.join(T2O.get(ch[binder][r],'X') for r in sorted(ch[binder]))
print(f"binder 서열 (직접 읽음): {seq}  ({len(ch[binder])}잔기)")

# manifest에서 원 서열도 확인
import csv as _csv
m={r['tag']:r['seq'] for r in _csv.DictReader(open(WORK+'/diversify_manifest.csv'))}
print(f"manifest 서열: {m.get(tag,'없음')}")

# PDB 뽑기 (binder=chain Z, target=A/B auth번호) — 좀 전 rebuild 로직
def o1(rl): return "".join(T2O.get(r,'X') for r in rl)
def fa(s,mo):
    o=[];i=s.find(mo)
    while i!=-1:o.append(i);i=s.find(mo,i+1)
    return o
def rebuild(src, dst):
    chd={}
    for ln in open(src):
        if ln[:4]=='ATOM' and ln[16] in (' ','A'):
            c=ln[21]; rs=int(ln[22:26])
            chd.setdefault(c,{}).setdefault(rs,[]).append(ln)
    b=min(chd,key=lambda c:len(chd[c])); out=[]
    for i,rs in enumerate(sorted(chd[b]),1):
        for ln in chd[b][rs]: out.append(ln[:21]+'Z'+f'{i:4d}'+ln[26:])
    out.append('TER\n'); pc=iter('AB')
    for c in chd:
        if c==b: continue
        order=sorted(chd[c]); s=o1([chd[c][r][0][17:20].strip() for r in order])
        nv=fa(s,"NAHV"); gg=fa(s,"GNSGG")
        for k,h in enumerate(nv):
            off=order[h+2]-220; nch=next(pc)
            lo=h if k==0 else (gg[k-1]+nv[k])//2
            hi=(gg[k]+nv[k+1])//2 if k+1<len(nv) else len(order)
            for idx in range(lo,hi):
                rs=order[idx]
                for ln in chd[c][rs]: out.append(ln[:21]+nch+f'{rs-off:4d}'+ln[26:])
            out.append('TER\n')
    out.append('END\n'); open(dst,'w').writelines(out)

os.makedirs(WORK+'/pure_junction_pdb',exist_ok=True)
dst=f"{WORK}/pure_junction_pdb/PUREJUNC_{seq}_{tag.replace('htra1_v3_','')}.pdb"
rebuild(p,dst)
shutil.make_archive(WORK+'/pure_junction_pdb','zip',WORK+'/pure_junction_pdb')
print(f"\n-> pure_junction_pdb.zip : {os.path.basename(dst)}")
print("ChimeraX: color /Z red / :301-314 blue / :285-290 green / :185-208 orange / :220,250,328 yellow")


순수 junction 1개의 tag/전체정보:
                           tag  seq  len site  loopA_n  contact_plddt  other_frac  cat_min     L3_res     LD_res
L12_htra1_v3_L12_30_model_3_s2  NaN   12   L3        2          81.13    0.315789     12.7 [305, 306] [287, 289]

tag = L12_htra1_v3_L12_30_model_3_s2
binder 서열 (직접 읽음): MTREQRDARERP  (12잔기)
manifest 서열: MTREQRDARERP

-> pure_junction_pdb.zip : PUREJUNC_MTREQRDARERP_L12_L12_30_model_3_s2.pdb
ChimeraX: color /Z red / :301-314 blue / :285-290 green / :185-208 orange / :220,250,328 yellow


In [ ]:
#@title 6. 인접잔기/거리 (L3-306 접촉 원자 · 상위 접촉 auth)
import glob, os, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK='/content/drive/MyDrive/htra1_v3_short'; OUT=WORK+'/diversify_out'
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        c=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        rn=ln[17:20].strip(); x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        ch.setdefault(c,{}).setdefault(rs,[rs,rn,{}])[2][an]=np.array([x,y,z])
    return {c:[d[k] for k in sorted(d)] for c,d in ch.items()}
def seq(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=seq(rl); nv=fa(s,"NAHV"); gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def off(rl):
    s=seq(rl); i=s.find("NAHV"); return rl[i+2][0]-220
print(f"{'tag':18s}| 최근접 | L3-306 접촉 | 상위 접촉 auth(<=5A)")
for f in sorted(glob.glob(OUT+"/*.pdb")):
    ch=parse(f); binder=min(ch,key=lambda c:len(ch[c]))
    ba=[(an,a) for r in ch[binder] for an,a in r[2].items()]
    P=prots([ch[c] for c in ch if c!=binder]); near=[]; best=None
    for p in P:
        o=off(p)
        for r in p:
            for tan,tc in r[2].items():
                for _,bc in ba: near.append((np.linalg.norm(tc-bc), r[0]-o))
        r306=next((r for r in p if r[0]-o==306),None)
        if r306:
            for tan,tc in r306[2].items():
                for ban,bc in ba:
                    d=np.linalg.norm(tc-bc)
                    if best is None or d<best[0]: best=(d,tan,ban)
    near.sort(); tag=os.path.basename(f).replace(".pdb","")
    top=sorted({a for d,a in near if d<=5.0})[:8]
    l3=(f"{round(float(best[0]),1)}A {best[1]}{'(OE)' if best[1] in('OE1','OE2') else ''}<->{best[2]}" if best else "-")
    print(f"{tag:18s}| {round(float(near[0][0]),1):>4}A | {l3:22s}| {top}")

In [ ]:
import glob, os, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
SEG={"LoopA":set(range(185,209)),"L3":set(range(301,315)),"LD":set(range(285,291)),"Loop2":set(range(345,353))}
CAT={220,250,328}; TIP={190,192,193,194,195,196,197}; ARG={190,197}
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        c=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        rn=ln[17:20].strip(); x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        ch.setdefault(c,{}).setdefault(rs,[rs,rn,{}])[2][an]=np.array([x,y,z])
    return {c:[d[k] for k in sorted(d)] for c,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=fa(s,"NAHV");gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None

print(f"{'tag':30s}| site  |junc|catCLR| LoopA L3  LD |arg| 접촉 auth")
for f in sorted(glob.glob(OUT+"/*.pdb")):
    ch=parse(f); binder=min(ch,key=lambda c:len(ch[c]))
    ba=np.array([a for r in ch[binder] for a in r[2].values()])
    P=prots([ch[c] for c in ch if c!=binder]); hits=[]
    for p in P:
        o=offof(p); h={}
        if o is not None:
            for r in p:
                auth=r[0]-o
                allA=list(r[2].values()); sc=[a for an,a in r[2].items() if an not in("N","CA","C","O")]
                da=min(float(np.linalg.norm(ba-a[None],axis=1).min()) for a in allA)
                ds=min((float(np.linalg.norm(ba-a[None],axis=1).min()) for a in sc),default=9e9)
                if da<=4.5: h[auth]=(da, ds<=4.5)
        hits.append(h)
    tally={k:0 for k in SEG}; tally["cat"]=0; tally["other"]=0
    for h in hits:
        for a,(d,sc) in h.items():
            w=1 if sc else 0.3; done=False
            for seg,rng in SEG.items():
                if a in rng: tally[seg]+=w; done=True; break
            if a in CAT: tally["cat"]+=w; done=True
            if not done: tally["other"]+=w
    site=max(tally,key=tally.get)
    junc=any(any(a in SEG["LD"] and sc for a,(d,sc) in hi.items()) and any(a in SEG["L3"] and sc for a,(d,sc) in hj.items()) for i,hi in enumerate(hits) for j,hj in enumerate(hits) if i!=j)
    catmin=9e9
    for p in P:
        o=offof(p)
        if o is None: continue
        for auth in CAT:
            r=next((x for x in p if x[0]-o==auth),None)
            if r:
                for a in r[2].values(): catmin=min(catmin,float(np.linalg.norm(ba-a[None],axis=1).min()))
    arg=any(a in ARG and sc for h in hits for a,(d,sc) in h.items())
    allauth=sorted(set(a for h in hits for a in h))
    tag=os.path.basename(f).replace(".pdb","").replace("htra1_v3_","")
    print(f"{tag:30s}| {site:5s}|{'Y' if junc else '.':^4}|{round(catmin,1):>5} |{tally['LoopA']:>5.1f}{tally['L3']:>4.1f}{tally['LD']:>4.1f} |{'Y' if arg else '.':^3}| {allauth}")

In [ ]:
import glob, os, numpy as np, csv, collections
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
SEG={"LoopA":set(range(185,209)),"L3":set(range(301,315)),"LD":set(range(285,291)),"Loop2":set(range(345,353))}
CAT={220,250,328}; ARG={190,197}
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        c=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        rn=ln[17:20].strip(); x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        ch.setdefault(c,{}).setdefault(rs,[rs,rn,{}])[2][an]=np.array([x,y,z])
    return {c:[d[k] for k in sorted(d)] for c,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=fa(s,"NAHV");gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl));
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None

rows=[]
for f in sorted(glob.glob(OUT+"/*.pdb")):
    ch=parse(f); binder=min(ch,key=lambda c:len(ch[c]))
    ba=np.array([a for r in ch[binder] for a in r[2].values()])
    P=prots([ch[c] for c in ch if c!=binder]); hits=[]
    for p in P:
        o=offof(p); h={}
        if o is not None:
            for r in p:
                an=list(r[2].keys()); T=np.array(list(r[2].values()))
                dmin=np.sqrt(((ba[:,None,:]-T[None,:,:])**2).sum(-1)).min(0)
                da=float(dmin.min()); sc=[i for i,a in enumerate(an) if a not in("N","CA","C","O")]
                ds=float(dmin[sc].min()) if sc else 9e9
                if da<=4.5: h[r[0]-o]=(round(da,1), ds<=4.5)
        hits.append(h)
    tally={k:0 for k in SEG}; tally["cat"]=0; tally["other"]=0
    for h in hits:
        for a,(d,sc) in h.items():
            w=1 if sc else 0.3; done=False
            for seg,rng in SEG.items():
                if a in rng: tally[seg]+=w;done=True;break
            if a in CAT: tally["cat"]+=w;done=True
            if not done: tally["other"]+=w
    site=max(tally,key=tally.get)
    junc=any(any(a in SEG["LD"] and sc for a,(d,sc) in hi.items()) and any(a in SEG["L3"] and sc for a,(d,sc) in hj.items()) for i,hi in enumerate(hits) for j,hj in enumerate(hits) if i!=j)
    catmin=9e9
    for p in P:
        o=offof(p)
        if o is None: continue
        for auth in CAT:
            r=next((x for x in p if x[0]-o==auth),None)
            if r: catmin=min(catmin,float(np.sqrt(((ba[:,None,:]-np.array(list(r[2].values()))[None,:,:])**2).sum(-1)).min()))
    arg=any(a in ARG and sc for h in hits for a,(d,sc) in h.items())
    allauth=sorted(set(a for h in hits for a in h))
    tag=os.path.basename(f).replace(".pdb","").replace("htra1_v3_","")
    bb="_".join(tag.split("_")[:-1])   # backbone (서열 s# 제거)
    rows.append(dict(tag=tag,bb=bb,site=site,junc=junc,catmin=round(catmin,1),ncontact=len(allauth),
                     LoopA=tally["LoopA"],L3=tally["L3"],LD=tally["LD"],arg=arg,contacts=allauth))

# 상세 CSV -> Drive
with open('/content/drive/MyDrive/htra1_v3_short/diversify_classified.csv','w',newline='') as fh:
    w=csv.DictWriter(fh,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

# 화면엔 backbone별 요약만
print(f"총 {len(rows)}개 판정. 상세 -> diversify_classified.csv\n")
print(f"{'backbone':28s}| n | site분포        | junc | ncontact(중앙) | catmin(중앙) | arg")
bbs=collections.OrderedDict()
for r in rows: bbs.setdefault(r['bb'],[]).append(r)
for bb,rs in bbs.items():
    sd=collections.Counter(r['site'] for r in rs)
    sdist=" ".join(f"{k}:{v}" for k,v in sd.most_common())
    nj=sum(1 for r in rs if r['junc'])
    ncmed=int(np.median([r['ncontact'] for r in rs]))
    cmed=round(float(np.median([r['catmin'] for r in rs])),1)
    narg=sum(1 for r in rs if r['arg'])
    print(f"{bb:28s}| {len(rs):>1} | {sdist:15s}| {nj}/{len(rs):<3}| {ncmed:>13} | {cmed:>11} | {narg}/{len(rs)}")

In [ ]:
import glob, os, numpy as np
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
OUT='/content/drive/MyDrive/htra1_v3_short/diversify_out'
TARGETS=['L12_htra1_v3_L12_150_model_2','L12_htra1_v3_L12_59_model_4','L06_htra1_v3_L06_140_model_7']
SEG={"LoopA":range(185,209),"L3":range(301,315),"LD":range(285,291),"Loop2":range(345,353)}
def segof(a):
    for s,r in SEG.items():
        if a in r: return s
    return "-"
T2O={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def parse(p):
    ch={}
    for ln in open(p):
        if not ln.startswith(("ATOM","HETATM")): continue
        if ln[16] not in (" ","A"): continue
        c=ln[21]
        try: rs=int(ln[22:26])
        except: continue
        an=ln[12:16].strip()
        if an.startswith("H") or an=="": continue
        rn=ln[17:20].strip(); x,y,z=float(ln[30:38]),float(ln[38:46]),float(ln[46:54])
        ch.setdefault(c,{}).setdefault(rs,[rs,rn,{}])[2][an]=np.array([x,y,z])
    return {c:[d[k] for k in sorted(d)] for c,d in ch.items()}
def o1(rl): return "".join(T2O.get(r[1],"X") for r in rl)
def fa(s,m):
    o=[];i=s.find(m)
    while i!=-1:o.append(i);i=s.find(m,i+1)
    return o
def prots(tc):
    P=[]
    for rl in tc:
        s=o1(rl);nv=fa(s,"NAHV");gg=fa(s,"GNSGG")
        if not nv: continue
        if len(nv)==1: P.append(rl)
        else:
            b=[0]
            for k in range(len(nv)-1):
                g=gg[k] if k<len(gg) else nv[k]; b.append((g+nv[k+1])//2)
            b.append(len(rl))
            for k in range(len(nv)): P.append(rl[b[k]:b[k+1]])
    return P
def offof(rl):
    s=o1(rl);ih=s.find("NAHV");iss=s.find("GNSGG")
    if ih<0 or iss<0: return None
    h=rl[ih+2][0]-220
    return h if h==rl[iss+2][0]-328 else None

for tgt in TARGETS:
    fs=sorted(glob.glob(f'{OUT}/{tgt}_s*.pdb'))
    if not fs: print(f'\n{tgt}: 파일 없음'); continue
    f=fs[0]  # 대표 1개(서열들 거의 동일)
    ch=parse(f); binder=min(ch,key=lambda c:len(ch[c]))
    ba=np.array([a for r in ch[binder] for a in r[2].values()])
    P=prots([ch[c] for c in ch if c!=binder])
    print(f'\n=== {tgt.replace("htra1_v3_","")}  (protomer {len(P)}개, 대표 {os.path.basename(f).split("_")[-1]}) ===')
    for pi,p in enumerate(P):
        o=offof(p)
        if o is None: continue
        hits=[]
        for r in p:
            auth=r[0]-o
            seg=segof(auth)
            if seg=="-" and auth not in (220,250,328): continue
            an=list(r[2].keys()); T=np.array(list(r[2].values()))
            dmin=np.sqrt(((ba[:,None,:]-T[None,:,:])**2).sum(-1)).min(0)
            da=float(dmin.min())
            sc=[i for i,x in enumerate(an) if x not in("N","CA","C","O")]
            ds=float(dmin[sc].min()) if sc else 9e9
            if da<=4.5:
                which="곁사슬" if ds<=4.5 else "백본만"
                hits.append(f"{seg}-{auth}({r[1]}) {round(da,1)}A[{which}]")
        if hits:
            print(f"  protomer {pi}: " + " | ".join(hits))
        else:
            print(f"  protomer {pi}: (조절요소 접촉 없음)")
    print("  >>> L3(305/306)와 LD(287)가 서로 다른 protomer의 [곁사슬]이면 = 진짜 inter-protomer clamp")